### Import libraries and set up the configuarations

In [ ]:
#### Importar librerias ####

from __future__ import division
from pyomo.environ import *
from pyomo.core import *
from pyomo.opt import SolverFactory
from datetime import datetime
from pathlib import Path
from itertools import product
from matplotlib import colors as mcolors
from matplotlib import cm
from matplotlib.colors import to_hex
from colorsys import rgb_to_hls, hls_to_rgb
from typing_extensions import final

import pandas as pd
import plotly.express as px
import numpy as np
import re  
import os
import itertools
import nbformat as nbf 

In [ ]:
# Configuration

model_name = 'col_regional'         # choose 'col_regional' or 'col_national'
div = 1                             # number of time slices (used for file naming only)

date_tag = datetime.now().strftime("%Y%m%d")    # e.g. 20260305

path_csv = "./CSV/"                 # input data directory (CSV format)
lp_dir = Path("./lp_files")        # folder to save LP files
lp_dir.mkdir(parents=True, exist_ok=True)

sol_dir = Path("./sol_files")   # where to save sol files
sol_dir.mkdir(parents=True, exist_ok=True)
#sol_file_name = model_name+"_"+date_tag+".sol"           # this is the general naming convention for solution files. You can change it if you want, but make sure to update the name in the code below where the solution file is read.
sol_file_name = "./sol_files/highs_solution.txt" # for this test we will use this solution file. 

# ======== Storage option ========
# Correr = "Storage"
Correr = "Normal"

# ======== Scenario option ========
Escenario = "Base"
# Escenario = "Carbononeutralidad"

# ======== Solver option ========
#Solver_selec = "glpk"
Solver_selec = "highs"

# ======== UDC option ========
usar_UDC = True


# ======== Output folders ========
tables_dir = Path("./outputs/tables")    # folder to save result tables (CSV)
graphs_dir = Path("./outputs/graphs")  # folder to save graphs (HTML)
for d in [tables_dir, graphs_dir]: d.mkdir(parents=True, exist_ok=True)


COLOR_FAMILIA = {
    "SOLAR": "#FDB813",
    "HIDRO": "#1F77B4",
    "EOLICA": "#2CA02C",
    "TERMICA_FOSIL": "#2B2B2B",
    "NUCLEAR": "#7B3F98", 
    "BIOMASA_RESIDUOS": "#8C6D31",
    "OTRAS": "#17BECF"
}


FAMILIAS_TEC = {
    "SOLAR": ["PWRSOLRTP", "PWRSOLUGE", "PWRSOLUPE", "PWRSOLRTP_ZNI"],
    "HIDRO": ["PWRHYDDAM", "PWRHYDROR", "PWRHYDROR_NDC"],
    "EOLICA": ["PWRWNDONS", "PWRWNDOFS_FIX", "PWRWNDOFS_FLO"],
    "TERMICA_FOSIL": ["PWRCOA",  "PWRCOACCS", "PWRNGS_CC", "PWRLPG", "PWRNGSCCS", "PWRNGS_SC"],
    "NUCLEAR": ["PWRNUC"],
    "BIOMASA_RESIDUOS": ["PWRAFR", "PWRBGS", "PWRWAS"],
    "OTRAS": ["PWRCSP", "PWRGEO", "PWRSTD"],
    }

#PWRNGSCCS, PWRNGS_SC  ; added to TERMICA_FOSIL family on 2024-06-05, as they are both fossil fuel technologies with carbon capture and storage.
#PWRSOLRTP_ZNI ; added to SOLAR family on 2024-06-05, as it is a solar technology with zero net imports (ZNI) feature. This technology was previously not categorized under any family, but it fits well within the SOLAR family based on its characteristics.



## Visualisation:
#### This script is devided per sectors (Power, Transport, Natural Gas, etc)
#### as of 13 March 2026, the power sector graphs were the focus, the rest are from the original code developed by David.R


In [ ]:
# Step1: Load Sets from CSV
## Loading sets from CSV to avoid using instance.model which consumes large memory. 

def load_sets(path_input):
    # This line ensures 'path' is a Path object even if you passed a string
    path = Path(path_input) 
    
    def get_list(file): 
        return pd.read_csv(path / file)["VALUE"].astype(str).tolist()
    
    sets = {
        "REGION": sorted(get_list("REGION.csv"), key=len, reverse=True),
        "TECHNOLOGY": sorted(get_list("TECHNOLOGY.csv"), key=len, reverse=True),
        "TIMESLICE": sorted(get_list("TIMESLICE.csv"), key=len, reverse=True),
        "FUEL": sorted(get_list("FUEL.csv"), key=len, reverse=True),
        "EMISSION": sorted(get_list("EMISSION.csv"), key=len, reverse=True),
        "YEAR": sorted([int(y) for y in get_list("YEAR.csv")]),
        "MODE_OF_OPERATION": sorted([int(float(y)) for y in get_list("MODE_OF_OPERATION.csv")]) 
    }
    return sets

sets = load_sets(path_csv)

In [ ]:
# Step2: Mapping power generation technologies to FAMILIAS_TEC and reading solution file 
# Note: FAMILIAS_TEC is introduced in the configuration section above. 

TECH_TO_FAMILY = {tech: fam for fam, techs in FAMILIAS_TEC.items() for tech in techs}

# PARSING LOGIC to read the HiGHS solution file outputs:

def smart_parse(index_str, sets):
    parts = index_str.split("_")
    # Cleaned up keys to match your standard OSeMOSYS dimensions
    data = {k: None for k in ["REGION", "TECHNOLOGY", "TIMESLICE", "YEAR", "MODE_OF_OPERATION", "FUEL", "EMISSION"]}
    
    # 1. Positional logic (Year is always last, MODE_OF_OPERATION is usually second to last)
    if parts[-1].isdigit(): 
        data["YEAR"] = int(parts[-1])
    
    # If the second to last part is a digit, it's our MODE_OF_OPERATION
    if len(parts) > 1 and parts[-2].isdigit(): 
        data["MODE_OF_OPERATION"] = int(parts[-2])
    
    # 2. Sliding window for sets (Tech, Fuel, Region, TS, Emission)
    for i in range(len(parts)):
        for j in range(i + 1, len(parts) + 1):
            chunk = "_".join(parts[i:j])
            
            if chunk in sets["TECHNOLOGY"]: data["TECHNOLOGY"] = chunk
            elif chunk in sets["TIMESLICE"]: data["TIMESLICE"] = chunk
            elif chunk in sets["REGION"]: data["REGION"] = chunk
            elif chunk in sets["FUEL"]: data["FUEL"] = chunk
            elif chunk in sets["EMISSION"]: data["EMISSION"] = chunk
                
    return pd.Series(data)

def read_highs_sol(filepath):
    rows = []
    # Regex to capture VarName(Index) Value
    pattern = re.compile(r'^(\w+)\((.+)\)\s+([\-\d\.eE\+]+)\s*$')
    with open(filepath, "r") as f:
        for line in f:
            m = pattern.match(line.strip())
            if m: rows.append((m.group(1), m.group(2), float(m.group(3))))
    return pd.DataFrame(rows, columns=["VarName", "IndexStr", "Value"])



In [ ]:
# # This is an additional 'DIAGNOSTIC & VERIFICATION STEP' to the HiGHS solution parsing. 

# def run_diagnostics(df_sol, sets):
#     print("\n" + "="*50)
#     print("HIGHS SOLUTION DIAGNOSTICS")
#     print("="*50)
    
#     # 1. List all variables and counts
#     var_counts = df_sol["VarName"].value_counts()
#     print("\n✅ Variables found in solution file:")
#     print(var_counts)
    
#     # 2. Check parsing accuracy on a sample
#     # We take the most common variable (usually RateOfActivity or Use)
#     sample_var = var_counts.index[0] 
#     print(f"\n🧪 Testing parser on sample of: {sample_var}")
    
#     test_sample = df_sol[df_sol["VarName"] == sample_var].head(5).copy()
#     parsed_sample = test_sample["IndexStr"].apply(lambda x: smart_parse(x, sets))
    
#     verification_df = pd.concat([test_sample[["IndexStr", "Value"]], parsed_sample], axis=1)
    
#     print("\nVerify that columns match the IndexStr below:")
#     print(verification_df.to_string(index=False))
    
#     # 3. Check for Unparsed Years or Technologies
#     # If any mandatory fields are None in the sample, it flags a warning
#     if parsed_sample["TECHNOLOGY"].isnull().any() and "Capacity" in sample_var:
#         print("\n⚠️ WARNING: Some technologies were not identified. Check TECHNOLOGY.csv names.")
    
#     print("="*50 + "\n")

# # To run it, just call it after you read the solution file in Step 5:
# df_raw = read_highs_sol(sol_file_name)
# run_diagnostics(df_raw, sets)

# # Ensure the engine also uses the lowercase version
# #engine = ResultsEngine(df_raw, sets, path_csv)

In [ ]:
# Step3:  CALCULATION ENGINE: since HiGHS generates certain decision varaiables, we need to calculate other varaiables like (productionbytechnology, etc)
# The initial code was changed using vectorized operations and groupby to improve performance.

## Step3a: Define the ResultsEngine class with vectorized calculations: 
## Note: more calculation methods can be added to this class as needed (e.g. emissions, costs)


class ResultsEngine:
    def __init__(self, df_sol, sets, csv_path):
        self.df_sol = df_sol
        self.sets = sets
        self.csv_path = Path(csv_path)

    def get_var(self, name):
        subset = self.df_sol[self.df_sol["VarName"] == name].copy()
        parsed = subset["IndexStr"].apply(lambda x: smart_parse(x, self.sets))
        return pd.concat([subset[["Value"]], parsed], axis=1)

    def add_subregions(self, df):
        """ add subregions from technology name"""
        df["SUBREGION"] = df["TECHNOLOGY"].apply(lambda x: x.split("_")[0])
        df["TECH_BASE"] = df["TECHNOLOGY"].apply(lambda x: "_".join(x.split("_")[1:]) if "_" in x else x)
        return df

    def calculate_production(self):
        roa = self.get_var("RateOfActivity")
        oar = pd.read_csv(self.csv_path / "OutputActivityRatio.csv")
        ys = pd.read_csv(self.csv_path / "YearSplit.csv")
        
        df = roa.merge(oar, on=["REGION", "TECHNOLOGY", "MODE_OF_OPERATION", "YEAR"])
        df = df.merge(ys, on=["TIMESLICE", "YEAR"])
        
        # from PJ -> TWh
        df["VALUE"] = (df["Value"] * df["VALUE_x"] * df["VALUE_y"]) / 3.6
        df = self.add_subregions(df)
        df2 = df.groupby(["REGION", "SUBREGION", "TECH_BASE", "YEAR"], as_index=False)["VALUE"].sum()
        return df2.rename(columns={"TECH_BASE": "TECHNOLOGY"})

    def calculate_new_capacity(self):
        nc = self.get_var("NewCapacity")
        nc["VALUE"] = nc["Value"] #/ 31.536  # Convert PJ to GW
        nc = self.add_subregions(nc)
        df2 = nc.groupby(["REGION", "SUBREGION", "TECH_BASE", "YEAR"], as_index=False)["VALUE"].sum()
        return df2.rename(columns={"TECH_BASE": "TECHNOLOGY"})

    def calculate_total_capacity(self):
        # 1. Get New Capacity and Residual
        nc = nc.groupby(["REGION", "TECHNOLOGY", "YEAR"], as_index=False)["NC_Raw"].sum() 
        rc = pd.read_csv(self.csv_path / "ResidualCapacity.csv").rename(columns={"VALUE": "RC_Value"})
        ol = pd.read_csv(self.csv_path / "OperationalLife.csv")
        
        # to ensure each tech only has ONE life value
        ol = ol.groupby(["REGION", "TECHNOLOGY"], as_index=False)["VALUE"].mean()
        


        # 2. Accumulate New Capacity based on Life
        all_years = sorted([int(y) for y in self.sets["YEAR"]])
        nc_full = nc.merge(pd.DataFrame({"YEAR_Eval": all_years}), how="cross")
        nc_full = nc_full.merge(ol, on=["REGION", "TECHNOLOGY"], how="left")

        # Filter: Investment is active if Evaluation Year is within [Year, Year + Life)
        active_mask = (nc_full["YEAR_Eval"] >= nc_full["YEAR"]) & \
                      (nc_full["YEAR_Eval"] < nc_full["YEAR"] + nc_full["VALUE"])
        
        acc_nc = nc_full[active_mask].groupby(["REGION", "TECHNOLOGY", "YEAR_Eval"], as_index=False)["NC_Raw"].sum()
        acc_nc = acc_nc.rename(columns={"YEAR_Eval": "YEAR", "NC_Raw": "ACC_NC"})

        # 3. Combine Accumulated New + Residual
        total = acc_nc.merge(rc, on=["REGION", "TECHNOLOGY", "YEAR"], how="outer").fillna(0)
        total["VALUE"] = (total["ACC_NC"] + total["RC_Value"]) #/ 31.536
        
        total = self.add_subregions(total)
        
        final = total.groupby(["REGION", "SUBREGION", "TECH_BASE", "YEAR"], as_index=False)["VALUE"].sum()
        return final.rename(columns={"TECH_BASE": "TECHNOLOGY"})
    
    

In [ ]:
# ## Test the calculation of total capacity for solar technology in 2022 to verify that the logic is working as expected.
# solar_test = df_total_cap[(df_total_cap['TECHNOLOGY'] == 'PWRSOLUGE') & (df_total_cap['YEAR'] == 2022)]
# print(f"DEBUG: Solar Capacity in 2022 is {solar_test['VALUE'].sum()} GW")

In [ ]:
## Step3b: Execute the engine to calculate production, new capacity, and total capacity tables:

print(f"Processing solution: {sol_file_name}")
df_raw = read_highs_sol(sol_file_name)
engine = ResultsEngine(df_raw, sets, path_csv)

# Generate Tables
df_prod = engine.calculate_production()
df_new_cap = engine.calculate_new_capacity()
df_total_cap = engine.calculate_total_capacity()

# Save Outputs
#df_prod.to_csv(tables_dir / f"ProductionByTechnology_{date_tag}.csv", index=False)
#df_new_cap.to_csv(tables_dir / f"NewCapacity_{date_tag}.csv", index=False)
#df_total_cap.to_csv(tables_dir / f"TotalCapacityAnnual_{date_tag}.csv", index=False)



In [ ]:
# Step4: PLOTTING:

def power_sector_plots(df, variable_label, unit="Units", level="National", subregion=None):
    """
    variable_label: e.g. 'ProductionByTechnology', 'NewCapacity', 'TotalCapacityAnnual'
    unit: e.g. 'TWh' or 'GW'
    """
    df_plot = df.copy()
    
    # --- SAFETY CHECK: Identify Missing Technologies ---
    all_techs = set(df_plot["TECHNOLOGY"].unique())
    mapped_techs = set(TECH_TO_FAMILY.keys())
    missing_techs = all_techs - mapped_techs
    
    print(f"\n--- 🛡️ Safety Check: {variable_label} ---")
    if missing_techs:
        print(f"⚠️ The following technologies were FILTERED OUT (not in FAMILIAS_TEC):")
        for t in sorted(missing_techs):
            print(f"   - {t}")
    else:
        print("✅ All technologies in the data are covered by your FAMILIAS_TEC dictionary.")

    # --- FILTERING ---
    # Only keep the ones we have families for
    df_plot = df_plot[df_plot["TECHNOLOGY"].isin(TECH_TO_FAMILY.keys())].copy()
    df_plot["FAMILY"] = df_plot["TECHNOLOGY"].map(TECH_TO_FAMILY)

    # --- AGGREGATION ---
    if level == "National":
        # Sum by Family across all subregions
        df_plot = df_plot.groupby(["FAMILY", "YEAR"], as_index=False)["VALUE"].sum()
        title = f"National - {variable_label} ({unit})"
    else:
        # Filter by subregion first
        df_plot = df_plot[df_plot["SUBREGION"] == subregion]
        # Sum by Family for that subregion
        df_plot = df_plot.groupby(["FAMILY", "YEAR"], as_index=False)["VALUE"].sum()
        title = f"{subregion} - {variable_label} ({unit})"

    # --- PLOT ---
    fig = px.bar(
        df_plot.sort_values("YEAR"), 
        x="YEAR", 
        y="VALUE", 
        color="FAMILY",
        title=title,
        color_discrete_map=COLOR_FAMILIA,
        template="plotly_white",
        labels={"VALUE": unit, "FAMILY": "Category"}
    )
    
    fig.update_layout(
        xaxis_type='category', 
        barmode='stack',
        legend_title_text='Technology',
    )
    fig.show()

# The User defines which graphs to generate: (National Vs Sub-regional level, and which variable to plot: production, new capacity, or total capacity)

# Example: Total Capacity at National Level
power_sector_plots(df_total_cap, "TotalCapacityAnnual", "GW", level="National")

# Example: Production at Sub-regional Level
power_sector_plots(df_prod, "ProductionByTechnology", "TWh", level="Sub-regional", subregion="CA")


In [ ]:
## Comments on the current plots: 
'''
- The graphs shows over production of solar at the national and sub-regional level. 
After debugging it is noticed that the model installs huge capcity of solar in the fist year (5170GW) which is not realistic. This could be due to several reasons such as:
1. low capital cost for solar,
2. missing capacity factor
3. missing total annual capacity constraint for solar. 

- once solar is removed from the graph the other technologies have reasonable capacities. 

'''

## GRAFICAR

#### *Gráficas Matriz Eléctrica* - *Power Sector Graphs*

#### Gas Natural

In [ ]:
###### Definir colores de los combustibles


def graf_GAS(variable_name, UN,solver_selec):

### Definir dataframe a graficar según solver

    if solver_selec == "highs":
        df_var = sol_variable_to_df(
            sol,
            variable_name,
            ['REGION', 'TIMESLICE', 'TECHNOLOGY', 'FUEL', 'YEAR']
        ).drop(columns="DUMMY")
        df_var.head(50)
    elif solver_selec == "glpk":
        variable_mapping = {
            # 'RateOfProductionByTechnology': instance.RateOfProductionByTechnology,
            # 'RateOfUseByTechnology': instance.RateOfUseByTechnology,
            'ProductionByTechnology': instance.ProductionByTechnology,
            # 'UseByTechnology': instance.UseByTechnology
        }
        if variable_name not in variable_mapping:
            raise ValueError(f"Variable no válida: {variable_name}")
        variable = variable_mapping[variable_name]
        df_var = variable_to_dataframe(variable, index_names=['REGION', 'TIMESLICE', 'TECHNOLOGY', 'FUEL', 'YEAR']) 

    df_cap = df_var.copy()
    if df_cap.empty:
        print("No hay datos")
        return

    # Combining YEAR and TIMESLICE into TIME
    df_cap['TIME'] = df_cap['YEAR'].astype(str) 

    # Combining TECHNOLOGY and FUEL into COLOR
    df_cap['COLOR'] = df_cap['TECHNOLOGY'].astype(str) 

    # Filtering only UPSREG or MINNGS and making a copy
    df_final = df_cap[(df_cap['TECHNOLOGY'].str.startswith("UPSREG")) | 
                      (df_cap['TECHNOLOGY'].str.startswith("MINNGS"))].copy()

    # Conversion factor: 1 Gpc ≈ 1.0095581216  PJ
    if UN == 'Gpc':
        df_final["VALUE"] = df_final["VALUE"] / 1.0095581216  # de PJ a Gpc segun BECO

    # Plotting with Plotly
    fig = px.bar(df_final, x="TIME", y="VALUE", color="COLOR", 
                 title='Oferta Gas Natural', labels={'VALUE': UN})
    fig.show()
    
    df_final.to_csv('resultados/'+'Prod_GN_'+UN+'.csv')


graf_GAS('ProductionByTechnology','PJ',Solver_selec)  ##Energía final
graf_GAS('ProductionByTechnology','Gpc',Solver_selec)  ##Energía final



def graf_GAS_consumo(variable_name, UN,solver_selec):

### Definir dataframe a graficar según solver

    if solver_selec == "highs":
        df_var = sol_variable_to_df(
            sol,
            variable_name,
            ['REGION', 'TIMESLICE', 'TECHNOLOGY', 'FUEL', 'YEAR']
        ).drop(columns="DUMMY")
        df_var.head(50)
    elif solver_selec == "glpk":
        variable_mapping = {
            # 'RateOfProductionByTechnology': instance.RateOfProductionByTechnology,
            # 'RateOfUseByTechnology': instance.RateOfUseByTechnology,
            # 'ProductionByTechnology': instance.ProductionByTechnology,
            'UseByTechnology': instance.UseByTechnology
        }
        if variable_name not in variable_mapping:
            raise ValueError(f"Variable no válida: {variable_name}")
        variable = variable_mapping[variable_name]
        df_var = variable_to_dataframe(variable, index_names=['REGION', 'TIMESLICE', 'TECHNOLOGY', 'FUEL', 'YEAR']) 

    df_cap = df_var.copy()
    if df_cap.empty:
        print("No hay datos")
        return
    # Combining YEAR and TIMESLICE into TIME
    df_cap['TIME'] = df_cap['YEAR'].astype(str) 

    # Combining TECHNOLOGY and FUEL into COLOR
    df_cap['COLOR'] = df_cap['TECHNOLOGY'].astype(str)

    # Filtering only NGS and making a copy
    df_final = df_cap[(df_cap['TECHNOLOGY'].str.contains("NGS"))].copy()

    # Conversion factor: 1 Gpc ≈ 1.0095581216  PJ
    if UN == 'Gpc':
        df_final["VALUE"] = df_final["VALUE"] / 1.0095581216  # de PJ a Gpc

    # Plotting with Plotly
    fig = px.bar(df_final, x="TIME", y="VALUE", color="COLOR", 
                 title='Consumo Gas Natural', labels={'VALUE': UN})
    fig.show()
    df_final.to_csv('resultados/'+'Consumo_GN_'+UN+'.csv')

graf_GAS_consumo('UseByTechnology','PJ',Solver_selec)  ##Consumo 
graf_GAS_consumo('UseByTechnology','Gpc',Solver_selec)  ##Consumo

#### REFINERIAS

In [ ]:
###### Definir colores de los combustibles

COLORES_GRUPOS = {
    'NGS': '#1f77b4',
    'JET': '#ff7f0e',
    'BGS': '#2ca02c',
    'BDL': '#d62728',
    'WAS': '#9467bd',
    'WOO': '#8c564b',
    'GSL': '#e377c2',
    'COA': '#7f7f7f',
    'ELC': '#bcbd22',
    'BAG': '#17becf',
    'DSL': '#aec7e8',
    'LPG': '#ffbb78',
    'PHEV': '#98df8a',
    'HEV': '#ff9896',
}

    # Agrupar por 'ELC'/ JET en la columna COLOR

def asignar_grupo(nombre):
    if 'ELC' in nombre:
        return 'ELC'
    elif 'JET' in nombre:
        return 'JET'
    elif 'LPG' in nombre:
        return 'LPG'
    elif 'BDL' in nombre:
        return 'BDL'
    elif 'WAS' in nombre:
        return 'WAS'
    elif 'BGS' in nombre:
        return 'BGS'
    elif 'GSL' in nombre:
        return 'GSL'
    elif 'DSL' in nombre:
        return 'DSL'
    elif 'NGS' in nombre:
        return 'NGS'
    elif 'BAG' in nombre:
        return 'BAG'
    elif 'COA' in nombre:
        return 'COA'
    elif 'FOL' in nombre:
        return 'FOL'
    elif 'APHEV' in nombre:
        return 'PHEV'
    elif 'AHEV' in nombre:
        return 'HEV' 
    else:
        return nombre



def graf_demand_REF_combustible_TOTAL(variable_name,UN,pasos,solver_selec):

### Definir dataframe a graficar según solver
    if solver_selec == "highs":
        df_var = sol_variable_to_df(
            sol,
            variable_name,
            ['REGION', 'TIMESLICE', 'TECHNOLOGY', 'FUEL', 'YEAR']
        ).drop(columns="DUMMY")
        df_var.head(50)
    elif solver_selec == "glpk":
        variable_mapping = {
            # 'RateOfProductionByTechnology': instance.RateOfProductionByTechnology,
            # 'RateOfUseByTechnology': instance.RateOfUseByTechnology,
            'ProductionByTechnology': instance.ProductionByTechnology,
            'UseByTechnology': instance.UseByTechnology
        }
        if variable_name not in variable_mapping:
            raise ValueError(f"Variable no válida: {variable_name}")
        variable = variable_mapping[variable_name]
        df_var = variable_to_dataframe(variable, index_names=['REGION', 'TIMESLICE', 'TECHNOLOGY', 'FUEL', 'YEAR']) 
        df_cap = df_var.copy()
        
    if df_cap.empty:
        print(f"No hay datos para la variable {variable_name}")
        return

    df_cap = df_cap[df_cap['TECHNOLOGY'].str.startswith('UPSREF')]


    if pasos ==1:
        df_cap['TIME'] = df_cap['YEAR'].astype(str)
    else:
        # Combining YEAR and TIMESLICE into TIME
        df_cap['TIME'] = df_cap['YEAR'].astype(str) + '_'+ df_cap['TIMESLICE']

    df_cap['COLOR'] = df_cap['TECHNOLOGY'] + '_' + df_cap['FUEL']

    df_cap['GROUP'] = df_cap['COLOR'].apply(asignar_grupo)

    df_final = df_cap.groupby(['GROUP', 'TIME'], as_index=False)['VALUE'].sum()
    df_final = df_final.rename(columns={'GROUP': 'COLOR'})

    df_final = df_final[df_final.groupby('COLOR')['VALUE'].transform('sum') > 1e-5]
    if UN=='TWh':
        df_final["VALUE"] /= 3.6  # Cambiar de PJ a TWh
    else:
        df_final["VALUE"] /= 1

    # Filtra solo los grupos presentes en df_final
    grupos_presentes = df_final["COLOR"].unique()

    # Construye la lista de colores en el orden correcto
    colores_final = [COLORES_GRUPOS.get(grupo, '#333333') for grupo in grupos_presentes]

    fig = px.bar(df_final, x="TIME", y="VALUE", color="COLOR",
                 title='REFINERIAS', labels={'VALUE': UN},
                 color_discrete_sequence=colores_final)
    # Ajustar la leyenda abajo
    # fig.update_layout(legend=dict(orientation="h", y=-0.4))
    fig.show()
    df_final.to_csv('resultados/'+'REFINERIAS_'+variable_name+'_'+UN+'.csv')

graf_demand_REF_combustible_TOTAL('ProductionByTechnology','PJ',div,Solver_selec)
graf_demand_REF_combustible_TOTAL('UseByTechnology','PJ',div,Solver_selec)

In [ ]:
### Refinerías + importaciones

def graf_demand_REF_combustible_IMPORT(variable_name,UN,pasos,solver_selec):
    
### Definir dataframe a graficar según solver
    if solver_selec == "highs":
        df_var = sol_variable_to_df(
            sol,
            variable_name,
            ['REGION', 'TIMESLICE', 'TECHNOLOGY', 'FUEL', 'YEAR']
        ).drop(columns="DUMMY")
        df_var.head(50)
    elif solver_selec == "glpk":
        variable_mapping = {
            # 'RateOfProductionByTechnology': instance.RateOfProductionByTechnology,
            # 'RateOfUseByTechnology': instance.RateOfUseByTechnology,
            'ProductionByTechnology': instance.ProductionByTechnology,
            'UseByTechnology': instance.UseByTechnology
        }
        if variable_name not in variable_mapping:
            raise ValueError(f"Variable no válida: {variable_name}")
        variable = variable_mapping[variable_name]
        df_var = variable_to_dataframe(variable, index_names=['REGION', 'TIMESLICE', 'TECHNOLOGY', 'FUEL', 'YEAR']) 
        df_cap = df_var.copy()

    if df_cap.empty:
        print(f"No hay datos para la variable {variable_name}")
        return

    df_cap = df_cap[df_cap['TECHNOLOGY'].str.startswith('UPSREF') | df_cap['TECHNOLOGY'].str.startswith('IMPLPG') | df_cap['TECHNOLOGY'].str.startswith('IMPDSL') | df_cap['TECHNOLOGY'].str.startswith('IMPGSL')]


    if pasos ==1:
        df_cap['TIME'] = df_cap['YEAR'].astype(str)
    else:
        # Combining YEAR and TIMESLICE into TIME
        df_cap['TIME'] = df_cap['YEAR'].astype(str) + '_'+ df_cap['TIMESLICE']

    df_cap['COLOR'] = df_cap['TECHNOLOGY'] 

    # df_cap['GROUP'] = df_cap['COLOR'].apply(asignar_grupo)

    # df_final = df_cap.groupby(['GROUP', 'TIME'], as_index=False)['VALUE'].sum()
    # df_final = df_final.rename(columns={'GROUP': 'COLOR'})

    # df_final = df_final[df_final.groupby('COLOR')['VALUE'].transform('sum') > 1e-5]
    if UN=='TWh':
        df_cap["VALUE"] /= 3.6  # Cambiar de PJ a TWh
    else:
        df_cap["VALUE"] /= 1

    # Filtra solo los grupos presentes en df_final
    # grupos_presentes = df_final["COLOR"].unique()

    # Construye la lista de colores en el orden correcto
    # colores_final = [COLORES_GRUPOS.get(grupo, '#333333') for grupo in grupos_presentes]

    fig = px.bar(df_cap, x="TIME", y="VALUE",
                 title='REFINERIAS', labels={'VALUE': UN}, color="COLOR", )
    # Ajustar la leyenda abajo
    # fig.update_layout(legend=dict(orientation="h", y=-0.4))
    fig.show()
    df_cap.to_csv('resultados/'+'REF+IMP_'+variable_name+'_'+UN+'.csv')
graf_demand_REF_combustible_IMPORT('ProductionByTechnology','PJ',div,Solver_selec)

#### RESIDENCIAL

In [ ]:
# Paleta base
COLORES_GRUPOS = {
    'NGS': '#1f77b4',
    'JET': '#ff7f0e',
    'BGS': '#2ca02c',
    'BDL': '#d62728',
    'WAS': '#9467bd',
    'WOO': '#8c564b',
    'GSL': '#e377c2',
    'COA': '#7f7f7f',
    'ELC': '#bcbd22',
    'BAG': '#17becf',
    'DSL': '#aec7e8',
    'LPG': '#ffbb78',
    'FOL': '#98df8a',
    'AUT': '#ff9896',
}


def graf_demand_RES_combustible_TOTAL(variable_name,UN,pasos,solver_selec):

### Definir dataframe a graficar según solver
    if solver_selec == "highs":
        df_var = sol_variable_to_df(
            sol,
            variable_name,
            ['REGION', 'TIMESLICE', 'TECHNOLOGY', 'FUEL', 'YEAR']
        ).drop(columns="DUMMY")
        df_var.head(50)
    elif solver_selec == "glpk":
        variable_mapping = {
            # 'RateOfProductionByTechnology': instance.RateOfProductionByTechnology,
            # 'RateOfUseByTechnology': instance.RateOfUseByTechnology,
            'ProductionByTechnology': instance.ProductionByTechnology,
            'UseByTechnology': instance.UseByTechnology
        }
        if variable_name not in variable_mapping:
            raise ValueError(f"Variable no válida: {variable_name}")
        variable = variable_mapping[variable_name]
        df_var = variable_to_dataframe(variable, index_names=['REGION', 'TIMESLICE', 'TECHNOLOGY', 'FUEL', 'YEAR']) 
        df_cap = df_var.copy()

    if df_cap.empty:
        print(f"No hay datos para la variable {variable_name}")
        return
    # Filtrar tecnologías que comienzan con 'DEMRES'
    df_cap = df_cap[df_cap['TECHNOLOGY'].str.startswith('DEMRES')]


    if pasos ==1:
        df_cap['TIME'] = df_cap['YEAR'].astype(str)
    else:
        # Combining YEAR and TIMESLICE into TIME
        df_cap['TIME'] = df_cap['YEAR'].astype(str) + '_'+ df_cap['TIMESLICE']

    df_cap['COLOR'] = df_cap['TECHNOLOGY'] + '_' + df_cap['FUEL']



    # Agrupar por 'ELC'/ JET en la columna COLOR

    def asignar_grupo(nombre):
        if 'ELC' in nombre:
            return 'ELC'
        elif 'JET' in nombre:
            return 'JET'
        elif 'LPG' in nombre:
            return 'LPG'
        elif 'BDL' in nombre:
            return 'BDL'
        elif 'WAS' in nombre:
            return 'WAS'
        elif 'BGS' in nombre:
            return 'BGS'
        elif 'GSL' in nombre:
            return 'GSL'
        elif 'DSL' in nombre:
            return 'DSL'
        elif 'NGS' in nombre:
            return 'NGS'
        elif 'BAG' in nombre:
            return 'BAG'
        elif 'COA' in nombre:
            return 'COA'
        elif 'WOO' in nombre:
            return 'WOO'
        elif 'FOL' in nombre:
            return 'FOL'
        elif 'AUT' in nombre:
            return 'AUT' 
        else:
            return nombre

    df_cap['GROUP'] = df_cap['COLOR'].apply(asignar_grupo)

    df_final = df_cap.groupby(['GROUP', 'TIME'], as_index=False)['VALUE'].sum()
    df_final = df_final.rename(columns={'GROUP': 'COLOR'})

    df_final = df_final[df_final.groupby('COLOR')['VALUE'].transform('sum') > 1e-5]
    if UN=='TWh':
        df_final["VALUE"] /= 3.6  # Cambiar de PJ a TWh
    else:
        df_final["VALUE"] /= 1

    # Filtra solo los grupos presentes en df_final
    grupos_presentes = df_final["COLOR"].unique()

    # Construye la lista de colores en el orden correcto
    colores_final = [COLORES_GRUPOS.get(grupo, '#333333') for grupo in grupos_presentes]

    fig = px.bar(df_final, x="TIME", y="VALUE", color="COLOR",
                title='SECTOR RESIDENCIAL - Consumo por combustible - ', labels={'VALUE': UN},
                color_discrete_sequence=colores_final)
    # Ajustar la leyenda abajo
    # fig.update_layout(legend=dict(orientation="h", y=-0.4))
    fig.show()


def graf_demand_RES_combustible(variable_name,UN,pasos,filtro,solver_selec):

### Definir dataframe a graficar según solver
    if solver_selec == "highs":
        df_var = sol_variable_to_df(
            sol,
            variable_name,
            ['REGION', 'TIMESLICE', 'TECHNOLOGY', 'FUEL', 'YEAR']
        ).drop(columns="DUMMY")
        df_var.head(50)
    elif solver_selec == "glpk":
        variable_mapping = {
            # 'RateOfProductionByTechnology': instance.RateOfProductionByTechnology,
            # 'RateOfUseByTechnology': instance.RateOfUseByTechnology,
            'ProductionByTechnology': instance.ProductionByTechnology,
            'UseByTechnology': instance.UseByTechnology
        }
        if variable_name not in variable_mapping:
            raise ValueError(f"Variable no válida: {variable_name}")
        variable = variable_mapping[variable_name]
        df_var = variable_to_dataframe(variable, index_names=['REGION', 'TIMESLICE', 'TECHNOLOGY', 'FUEL', 'YEAR']) 
        df_cap = df_var.copy()

    if df_cap.empty:
        print(f"No hay datos para la variable {variable_name}")
        return

    df_cap = df_cap[df_cap['TECHNOLOGY'].str.startswith('DEMRES') & df_cap['TECHNOLOGY'].str.contains(filtro)]


    if pasos ==1:
        df_cap['TIME'] = df_cap['YEAR'].astype(str)
    else:
        # Combining YEAR and TIMESLICE into TIME
        df_cap['TIME'] = df_cap['YEAR'].astype(str) + '_'+ df_cap['TIMESLICE']

    df_cap['COLOR'] = df_cap['TECHNOLOGY'] + '_' + df_cap['FUEL']

    # Agrupar por 'ELC'/ JET en la columna COLOR

    def asignar_grupo(nombre):
        if 'ELC' in nombre:
            return 'ELC'
        elif 'JET' in nombre:
            return 'JET'
        elif 'LPG' in nombre:
            return 'LPG'
        elif 'BDL' in nombre:
            return 'BDL'
        elif 'WAS' in nombre:
            return 'WAS'
        elif 'BGS' in nombre:
            return 'BGS'
        elif 'GSL' in nombre:
            return 'GSL'
        elif 'DSL' in nombre:
            return 'DSL'
        elif 'NGS' in nombre:
            return 'NGS'
        elif 'BAG' in nombre:
            return 'BAG'
        elif 'COA' in nombre:
            return 'COA'
        elif 'WOO' in nombre:
            return 'WOO'
        elif 'FOL' in nombre:
            return 'FOL'
        elif 'AUT' in nombre:
            return 'AUT' 
        else:
            return nombre

    df_cap['GROUP'] = df_cap['COLOR'].apply(asignar_grupo)

    df_final = df_cap.groupby(['GROUP', 'TIME'], as_index=False)['VALUE'].sum()
    df_final = df_final.rename(columns={'GROUP': 'COLOR'})

    df_final = df_final[df_final.groupby('COLOR')['VALUE'].transform('sum') > 1e-5]
    if UN=='TWh':
        df_final["VALUE"] /= 3.6  # Cambiar de PJ a TWh
    else:
        df_final["VALUE"] /= 1

    # Filtra solo los grupos presentes en df_final
    grupos_presentes = df_final["COLOR"].unique()

    # Construye la lista de colores en el orden correcto
    colores_final = [COLORES_GRUPOS.get(grupo, '#333333') for grupo in grupos_presentes]

    fig = px.bar(df_final, x="TIME", y="VALUE", color="COLOR",
                title='SECTOR RESIDENCIAL - Consumo por combustible - ' + filtro, labels={'VALUE': UN},
                color_discrete_sequence=colores_final)
    # Ajustar la leyenda abajo
    # fig.update_layout(legend=dict(orientation="h", y=-0.4))
    fig.show()
    df_final.to_csv('resultados/'+'RESIDENCIAL_'+filtro+'_'+variable_name+'_'+UN+'.csv')

def asignar_grupo(nombre):
    for grupo in COLORES_GRUPOS:
        if grupo in nombre:
            return grupo
    return 'OTRO'


def generar_colores_tecnologias(df, columna='COLOR'):
    df['GRUPO'] = df[columna].apply(asignar_grupo)
    color_dict = {}
    orden_final = []

    for grupo in sorted(df['GRUPO'].unique()):
        subitems = sorted(df[df['GRUPO'] == grupo][columna].unique())
        base_color = COLORES_GRUPOS.get(grupo, '#999999')
        rgb = mcolors.to_rgb(base_color)
        h, l, s = rgb_to_hls(*rgb)
        n = len(subitems)

        for i, item in enumerate(subitems):
            if n <= 3:
                # Variaciones simples de luminosidad
                factor = 0.6 + 0.2 * (i / max(n - 1, 1))
                l_adj = min(max(l * factor, 0), 1)
                new_rgb = hls_to_rgb(h, l_adj, s)
            else:
                # Variar matiz y luminosidad ligeramente
                hue_shift = (i / n) * 0.15  # solo un poco alrededor del matiz base
                lightness_shift = 0.45 + 0.4 * (i / (n - 1))
                new_rgb = hls_to_rgb((h + hue_shift) % 1.0, lightness_shift, s)

            color_dict[item] = to_hex(np.clip(new_rgb, 0, 1))
            orden_final.append(item)

    return [color_dict[c] for c in orden_final], orden_final

def graf_demand_RES_TEC(variable_name, UN, pasos,filtro,LOC,solver_selec):

### Definir dataframe a graficar según solver
    if solver_selec == "highs":
        df_var = sol_variable_to_df(
            sol,
            variable_name,
            ['REGION', 'TIMESLICE', 'TECHNOLOGY', 'FUEL', 'YEAR']
        ).drop(columns="DUMMY")
        df_var.head(50)
    elif solver_selec == "glpk":
        variable_mapping = {
            # 'RateOfProductionByTechnology': instance.RateOfProductionByTechnology,
            # 'RateOfUseByTechnology': instance.RateOfUseByTechnology,
            'ProductionByTechnology': instance.ProductionByTechnology,
            'UseByTechnology': instance.UseByTechnology
        }
        if variable_name not in variable_mapping:
            raise ValueError(f"Variable no válida: {variable_name}")
        variable = variable_mapping[variable_name]
        df_var = variable_to_dataframe(variable, index_names=['REGION', 'TIMESLICE', 'TECHNOLOGY', 'FUEL', 'YEAR']) 
        df_cap = df_var.copy()

    if df_cap.empty:
        print(f"No hay datos para la variable {variable_name}")
        return

    if LOC=='URB':
        string='URBANO'
        df_cap = df_cap[
            df_cap['TECHNOLOGY'].str.startswith('DEMRES') &
            df_cap['TECHNOLOGY'].str.contains(filtro) &
            ~df_cap['TECHNOLOGY'].str.contains('RUR') &
            ~df_cap['TECHNOLOGY'].str.contains('ZNI')
        ]
    elif LOC=='RUR':
        string='RURAL'
        df_cap = df_cap[
            df_cap['TECHNOLOGY'].str.startswith('DEMRES') &
            df_cap['TECHNOLOGY'].str.contains(filtro) &
            df_cap['TECHNOLOGY'].str.contains('RUR') &
            ~df_cap['TECHNOLOGY'].str.contains('ZNI')
        ]
    else:
        string='ZNI'
        df_cap = df_cap[
            df_cap['TECHNOLOGY'].str.startswith('DEMRES') &
            df_cap['TECHNOLOGY'].str.contains(filtro) &
            ~df_cap['TECHNOLOGY'].str.contains('RUR') &
            df_cap['TECHNOLOGY'].str.contains('ZNI')
        ]       

    df_cap['TIME'] = df_cap['YEAR'].astype(str) if pasos == 1 else df_cap['YEAR'].astype(str) + '_' + df_cap['TIMESLICE']
    df_cap['COLOR'] = df_cap['TECHNOLOGY'] 

    # Eliminar entradas insignificantes
    df_cap = df_cap[df_cap.groupby('COLOR')['VALUE'].transform('sum') > 1e-5]

    df_final = df_cap.copy()
    if UN == 'TWh':
        df_final["VALUE"] /= 3.6  # De PJ a TWh

    # Aplicar colores personalizados
    colores_final, orden_color = generar_colores_tecnologias(df_final, 'COLOR')

    fig = px.bar(df_final, x="TIME", y="VALUE", color="COLOR",
                title='Consumo por tecnología '+filtro+' '+string, labels={'VALUE': UN},
                color_discrete_sequence=colores_final,
                category_orders={'COLOR': orden_color})  

    fig.show()
    df_final.to_csv('resultados/'+'RESIDENCIAL_'+filtro+'_'+variable_name+'_'+UN+'.csv')

graf_demand_RES_combustible_TOTAL('UseByTechnology','PJ',1,Solver_selec)

graf_demand_RES_combustible('UseByTechnology','PJ',1,'CKN',Solver_selec)

# graf_demand_RES_TEC('ProductionByTechnology','PJ',1,'CKN','URB')
# graf_demand_RES_TEC('ProductionByTechnology','PJ',1,'CKN','RUR')
# graf_demand_RES_TEC('ProductionByTechnology','PJ',1,'CKN','ZNI')

graf_demand_RES_combustible('UseByTechnology','PJ',1,'WHT',Solver_selec)

# graf_demand_RES_TEC('ProductionByTechnology','PJ',1,'WHT','URB')
# graf_demand_RES_TEC('ProductionByTechnology','PJ',1,'WHT','RUR')


graf_demand_RES_combustible('UseByTechnology','PJ',1,'AIR',Solver_selec)

# graf_demand_RES_TEC('ProductionByTechnology','PJ',1,'AIR','URB')
# graf_demand_RES_TEC('ProductionByTechnology','PJ',1,'AIR','RUR')

graf_demand_RES_combustible('UseByTechnology','PJ',1,'FAN',Solver_selec)

# graf_demand_RES_TEC('ProductionByTechnology','PJ',1,'FAN','URB')
# graf_demand_RES_TEC('ProductionByTechnology','PJ',1,'FAN','RUR')

graf_demand_RES_combustible('UseByTechnology','PJ',1,'REF',Solver_selec)

graf_demand_RES_TEC('ProductionByTechnology','PJ',1,'REF','URB',Solver_selec)
# graf_demand_RES_TEC('ProductionByTechnology','PJ',1,'REF','RUR')

graf_demand_RES_combustible('UseByTechnology','PJ',1,'TV',Solver_selec)

graf_demand_RES_TEC('UseByTechnology','PJ',1,'TV','URB',Solver_selec)
graf_demand_RES_TEC('UseByTechnology','PJ',1,'TV','RUR',Solver_selec)

graf_demand_RES_combustible('UseByTechnology','PJ',1,'WSH',Solver_selec)
graf_demand_RES_combustible('UseByTechnology','PJ',1,'OTH',Solver_selec)

#### INDUSTRIA

In [ ]:
###### Definir colores de los combustibles

COLORES_GRUPOS = {
    'NGS': '#1f77b4',
    'JET': '#ff7f0e',
    'BGS': '#2ca02c',
    'BDL': '#d62728',
    'WAS': '#9467bd',
    'WOO': '#8c564b',
    'GSL': '#e377c2',
    'COA': '#7f7f7f',
    'ELC': '#bcbd22',
    'BAG': '#17becf',
    'DSL': '#aec7e8',
    'LPG': '#ffbb78',
    'FOL': '#98df8a',
    'AUT': '#ff9896',
}


def graf_demand_IND_combustible_TOTAL(variable_name,UN,pasos,solver_selec):
### Definir dataframe a graficar según solver
    if solver_selec == "highs":
        df_var = sol_variable_to_df(
            sol,
            variable_name,
            ['REGION', 'TIMESLICE', 'TECHNOLOGY', 'FUEL', 'YEAR']
        ).drop(columns="DUMMY")
        df_var.head(50)
    elif solver_selec == "glpk":
        variable_mapping = {
            # 'RateOfProductionByTechnology': instance.RateOfProductionByTechnology,
            # 'RateOfUseByTechnology': instance.RateOfUseByTechnology,
            'ProductionByTechnology': instance.ProductionByTechnology,
            'UseByTechnology': instance.UseByTechnology
        }
        if variable_name not in variable_mapping:
            raise ValueError(f"Variable no válida: {variable_name}")
        variable = variable_mapping[variable_name]
        df_var = variable_to_dataframe(variable, index_names=['REGION', 'TIMESLICE', 'TECHNOLOGY', 'FUEL', 'YEAR']) 
        df_cap = df_var.copy()

    if df_cap.empty:
        print(f"No hay datos para la variable {variable_name}")
        return

    df_cap = df_cap[df_cap['TECHNOLOGY'].str.startswith('DEMIND')]


    if pasos ==1:
        df_cap['TIME'] = df_cap['YEAR'].astype(str)
    else:
        # Combining YEAR and TIMESLICE into TIME
        df_cap['TIME'] = df_cap['YEAR'].astype(str) + '_'+ df_cap['TIMESLICE']

    df_cap['COLOR'] = df_cap['TECHNOLOGY'] + '_' + df_cap['FUEL']



    # Agrupar por 'ELC'/ JET en la columna COLOR

    def asignar_grupo(nombre):
        if 'ELC' in nombre:
            return 'ELC'
        elif 'JET' in nombre:
            return 'JET'
        elif 'LPG' in nombre:
            return 'LPG'
        elif 'BDL' in nombre:
            return 'BDL'
        elif 'WAS' in nombre:
            return 'WAS'
        elif 'BGS' in nombre:
            return 'BGS'
        elif 'GSL' in nombre:
            return 'GSL'
        elif 'DSL' in nombre:
            return 'DSL'
        elif 'NGS' in nombre:
            return 'NGS'
        elif 'BAG' in nombre:
            return 'BAG'
        elif 'COA' in nombre:
            return 'COA'
        elif 'WOO' in nombre:
            return 'WOO'
        elif 'FOL' in nombre:
            return 'FOL'
        elif 'AUT' in nombre:
            return 'AUT' 
        else:
            return nombre

    df_cap['GROUP'] = df_cap['COLOR'].apply(asignar_grupo)

    df_final = df_cap.groupby(['GROUP', 'TIME'], as_index=False)['VALUE'].sum()
    df_final = df_final.rename(columns={'GROUP': 'COLOR'})

    df_final = df_final[df_final.groupby('COLOR')['VALUE'].transform('sum') > 1e-5]
    if UN=='TWh':
        df_final["VALUE"] /= 3.6  # Cambiar de PJ a TWh
    else:
        df_final["VALUE"] /= 1

    # Filtra solo los grupos presentes en df_final
    grupos_presentes = df_final["COLOR"].unique()

    # Construye la lista de colores en el orden correcto
    colores_final = [COLORES_GRUPOS.get(grupo, '#333333') for grupo in grupos_presentes]

    fig = px.bar(df_final, x="TIME", y="VALUE", color="COLOR",
                 title='SECTOR INDUSTRIAL - Consumo por combustible - ', labels={'VALUE': UN},
                 color_discrete_sequence=colores_final)
    # Ajustar la leyenda abajo
    # fig.update_layout(legend=dict(orientation="h", y=-0.4))
    fig.show()
    df_final.to_csv('resultados/'+'INDUSTRIAL TOTAL_'+variable_name+'_'+UN+'.csv')


def graf_demand_ind_combustible(variable_name,UN,pasos,filtro,solver_selec):
### Definir dataframe a graficar según solver
    if solver_selec == "highs":
        df_var = sol_variable_to_df(
            sol,
            variable_name,
            ['REGION', 'TIMESLICE', 'TECHNOLOGY', 'FUEL', 'YEAR']
        ).drop(columns="DUMMY")
        df_var.head(50)
    elif solver_selec == "glpk":
        variable_mapping = {
            # 'RateOfProductionByTechnology': instance.RateOfProductionByTechnology,
            # 'RateOfUseByTechnology': instance.RateOfUseByTechnology,
            'ProductionByTechnology': instance.ProductionByTechnology,
            'UseByTechnology': instance.UseByTechnology
        }
        if variable_name not in variable_mapping:
            raise ValueError(f"Variable no válida: {variable_name}")
        variable = variable_mapping[variable_name]
        df_var = variable_to_dataframe(variable, index_names=['REGION', 'TIMESLICE', 'TECHNOLOGY', 'FUEL', 'YEAR']) 
        df_cap = df_var.copy()

    if df_cap.empty:
        print(f"No hay datos para la variable {variable_name}")
        return

    df_cap = df_cap[df_cap['TECHNOLOGY'].str.startswith('DEMIND') & df_cap['TECHNOLOGY'].str.contains(filtro)]


    if pasos ==1:
        df_cap['TIME'] = df_cap['YEAR'].astype(str)
    else:
        # Combining YEAR and TIMESLICE into TIME
        df_cap['TIME'] = df_cap['YEAR'].astype(str) + '_'+ df_cap['TIMESLICE']

    df_cap['COLOR'] = df_cap['TECHNOLOGY'] + '_' + df_cap['FUEL']


    # Agrupar por 'ELC'/ JET en la columna COLOR

    def asignar_grupo(nombre):
        if 'ELC' in nombre:
            return 'ELC'
        elif 'JET' in nombre:
            return 'JET'
        elif 'LPG' in nombre:
            return 'LPG'
        elif 'BDL' in nombre:
            return 'BDL'
        elif 'WAS' in nombre:
            return 'WAS'
        elif 'BGS' in nombre:
            return 'BGS'
        elif 'GSL' in nombre:
            return 'GSL'
        elif 'DSL' in nombre:
            return 'DSL'
        elif 'NGS' in nombre:
            return 'NGS'
        elif 'BAG' in nombre:
            return 'BAG'
        elif 'COA' in nombre:
            return 'COA'
        elif 'WOO' in nombre:
            return 'WOO'
        elif 'FOL' in nombre:
            return 'FOL'
        elif 'AUT' in nombre:
            return 'AUT' 
        else:
            return nombre

    df_cap['GROUP'] = df_cap['COLOR'].apply(asignar_grupo)

    df_final = df_cap.groupby(['GROUP', 'TIME'], as_index=False)['VALUE'].sum()
    df_final = df_final.rename(columns={'GROUP': 'COLOR'})

    df_final = df_final[df_final.groupby('COLOR')['VALUE'].transform('sum') > 1e-5]
    if UN=='TWh':
        df_final["VALUE"] /= 3.6  # Cambiar de PJ a TWh
    else:
        df_final["VALUE"] /= 1

    # Filtra solo los grupos presentes en df_final
    grupos_presentes = df_final["COLOR"].unique()

    # Construye la lista de colores en el orden correcto
    colores_final = [COLORES_GRUPOS.get(grupo, '#333333') for grupo in grupos_presentes]

    fig = px.bar(df_final, x="TIME", y="VALUE", color="COLOR",
                 title='Consumo por combustible Industria ' + filtro, labels={'VALUE': UN},
                 color_discrete_sequence=colores_final)
    # Ajustar la leyenda abajo
    # fig.update_layout(legend=dict(orientation="h", y=-0.4))
    fig.show()
    df_final.to_csv('resultados/'+'INDUSTRIAL_'+filtro+'_'+variable_name+'_'+UN+'.csv')

graf_demand_IND_combustible_TOTAL('UseByTechnology','PJ',1,Solver_selec)

graf_demand_ind_combustible('ProductionByTechnology','PJ',1,'BOI',Solver_selec)
graf_demand_ind_combustible('ProductionByTechnology','PJ',1,'FUR',Solver_selec)
graf_demand_ind_combustible('ProductionByTechnology','PJ',1,'MPW',Solver_selec)
graf_demand_ind_combustible('ProductionByTechnology','PJ',1,'AIR',Solver_selec)
graf_demand_ind_combustible('ProductionByTechnology','PJ',1,'REF',Solver_selec)
graf_demand_ind_combustible('ProductionByTechnology','PJ',1,'ILU',Solver_selec)
graf_demand_ind_combustible('ProductionByTechnology','PJ',1,'OTH',Solver_selec)

#### TRANSPORTE

In [ ]:
###### Definir colores de los combustibles

COLORES_GRUPOS = {
    'NGS': '#1f77b4',
    'JET': '#ff7f0e',
    'BGS': '#2ca02c',
    'BDL': '#d62728',
    'WAS': '#9467bd',
    'WOO': '#8c564b',
    'GSL': '#e377c2',
    'COA': '#7f7f7f',
    'ELC': '#bcbd22',
    'BAG': '#17becf',
    'DSL': '#aec7e8',
    'LPG': '#ffbb78',
    'PHEV': '#98df8a',
    'HEV': '#ff9896',
}

    # Agrupar por 'ELC'/ JET en la columna COLOR

def asignar_grupo(nombre):
    if 'ELC' in nombre:
        return 'ELC'
    elif 'JET' in nombre:
        return 'JET'
    elif 'LPG' in nombre:
        return 'LPG'
    elif 'BDL' in nombre:
        return 'BDL'
    elif 'WAS' in nombre:
        return 'WAS'
    elif 'BGS' in nombre:
        return 'BGS'
    elif 'GSL' in nombre:
        return 'GSL'
    elif 'DSL' in nombre:
        return 'DSL'
    elif 'NGS' in nombre:
        return 'NGS'
    elif 'BAG' in nombre:
        return 'BAG'
    elif 'COA' in nombre:
        return 'COA'
    elif 'FOL' in nombre:
        return 'FOL'
    elif 'APHEV' in nombre:
        return 'PHEV'
    elif 'AHEV' in nombre:
        return 'HEV' 
    else:
        return nombre



def graf_demand_TRA_combustible_TOTAL(variable_name,UN,pasos,solver_selec):
### Definir dataframe a graficar según solver
    if solver_selec == "highs":
        df_var = sol_variable_to_df(
            sol,
            variable_name,
            ['REGION', 'TIMESLICE', 'TECHNOLOGY', 'FUEL', 'YEAR']
        ).drop(columns="DUMMY")
        df_var.head(50)
    elif solver_selec == "glpk":
        variable_mapping = {
            # 'RateOfProductionByTechnology': instance.RateOfProductionByTechnology,
            # 'RateOfUseByTechnology': instance.RateOfUseByTechnology,
            'ProductionByTechnology': instance.ProductionByTechnology,
            'UseByTechnology': instance.UseByTechnology
        }
        if variable_name not in variable_mapping:
            raise ValueError(f"Variable no válida: {variable_name}")
        variable = variable_mapping[variable_name]
        df_var = variable_to_dataframe(variable, index_names=['REGION', 'TIMESLICE', 'TECHNOLOGY', 'FUEL', 'YEAR']) 
        df_cap = df_var.copy()

    if df_cap.empty:
        print(f"No hay datos para la variable {variable_name}")
        return

    df_cap = df_cap[df_cap['TECHNOLOGY'].str.startswith('DEMTRA')]


    if pasos ==1:
        df_cap['TIME'] = df_cap['YEAR'].astype(str)
    else:
        # Combining YEAR and TIMESLICE into TIME
        df_cap['TIME'] = df_cap['YEAR'].astype(str) + '_'+ df_cap['TIMESLICE']

    df_cap['COLOR'] = df_cap['TECHNOLOGY'] + '_' + df_cap['FUEL']

    df_cap['GROUP'] = df_cap['COLOR'].apply(asignar_grupo)

    df_final = df_cap.groupby(['GROUP', 'TIME'], as_index=False)['VALUE'].sum()
    df_final = df_final.rename(columns={'GROUP': 'COLOR'})

    df_final = df_final[df_final.groupby('COLOR')['VALUE'].transform('sum') > 1e-5]
    if UN=='TWh':
        df_final["VALUE"] /= 3.6  # Cambiar de PJ a TWh
    else:
        df_final["VALUE"] /= 1

    # Filtra solo los grupos presentes en df_final
    grupos_presentes = df_final["COLOR"].unique()

    # Construye la lista de colores en el orden correcto
    colores_final = [COLORES_GRUPOS.get(grupo, '#333333') for grupo in grupos_presentes]

    fig = px.bar(df_final, x="TIME", y="VALUE", color="COLOR",
                 title='SECTOR TRANSPORTE - Consumo por combustible - '+variable_name, labels={'VALUE': UN},
                 color_discrete_sequence=colores_final)
    # Ajustar la leyenda abajo
    # fig.update_layout(legend=dict(orientation="h", y=-0.4))
    fig.show()
    df_final.to_csv('resultados/'+'TRANSPORTE TOTAL_'+variable_name+'_'+UN+'.csv')


def graf_demand_TRA_combustible(variable_name,UN,pasos,filtro,solver_selec):
### Definir dataframe a graficar según solver
    if solver_selec == "highs":
        df_var = sol_variable_to_df(
            sol,
            variable_name,
            ['REGION', 'TIMESLICE', 'TECHNOLOGY', 'FUEL', 'YEAR']
        ).drop(columns="DUMMY")
        df_var.head(50)
    elif solver_selec == "glpk":
        variable_mapping = {
            # 'RateOfProductionByTechnology': instance.RateOfProductionByTechnology,
            # 'RateOfUseByTechnology': instance.RateOfUseByTechnology,
            'ProductionByTechnology': instance.ProductionByTechnology,
            'UseByTechnology': instance.UseByTechnology
        }
        if variable_name not in variable_mapping:
            raise ValueError(f"Variable no válida: {variable_name}")
        variable = variable_mapping[variable_name]
        df_var = variable_to_dataframe(variable, index_names=['REGION', 'TIMESLICE', 'TECHNOLOGY', 'FUEL', 'YEAR']) 
        df_cap = df_var.copy()

    if df_cap.empty:
        print(f"No hay datos para la variable {variable_name}")
        return

    df_cap = df_cap[df_cap['TECHNOLOGY'].str.startswith('DEMTRA') & df_cap['TECHNOLOGY'].str.contains(filtro)]


    if pasos ==1:
        df_cap['TIME'] = df_cap['YEAR'].astype(str)
    else:
        # Combining YEAR and TIMESLICE into TIME
        df_cap['TIME'] = df_cap['YEAR'].astype(str) + '_'+ df_cap['TIMESLICE']

    df_cap['COLOR'] = df_cap['TECHNOLOGY'] + '_' + df_cap['FUEL']



    df_cap['GROUP'] = df_cap['COLOR'].apply(asignar_grupo)

    df_final = df_cap.groupby(['GROUP', 'TIME'], as_index=False)['VALUE'].sum()
    df_final = df_final.rename(columns={'GROUP': 'COLOR'})

    df_final = df_final[df_final.groupby('COLOR')['VALUE'].transform('sum') > 1e-5]
    if UN=='TWh':
        df_final["VALUE"] /= 3.6  # Cambiar de PJ a TWh
    else:
        df_final["VALUE"] /= 1

    # Filtra solo los grupos presentes en df_final
    grupos_presentes = df_final["COLOR"].unique()

    # Construye la lista de colores en el orden correcto
    colores_final = [COLORES_GRUPOS.get(grupo, '#333333') for grupo in grupos_presentes]

    fig = px.bar(df_final, x="TIME", y="VALUE", color="COLOR",
                 title='SECTOR TRANSPORTE - Consumo por combustible- ' + filtro, labels={'VALUE': UN},
                 color_discrete_sequence=colores_final)
    # Ajustar la leyenda abajo
    # fig.update_layout(legend=dict(orientation="h", y=-0.4))
    fig.show()
    df_final.to_csv('resultados/'+'TRANSPORTE_'+filtro+'_'+variable_name+'_'+UN+'.csv')

graf_demand_TRA_combustible_TOTAL('UseByTechnology','PJ',1,Solver_selec)
graf_demand_TRA_combustible_TOTAL('ProductionByTechnology','PJ',1,Solver_selec)

graf_demand_TRA_combustible('ProductionByTechnology','PJ',1,'MOT',Solver_selec)
graf_demand_TRA_combustible('ProductionByTechnology','PJ',1,'LDV',Solver_selec)
graf_demand_TRA_combustible('ProductionByTechnology','PJ',1,'FWD',Solver_selec)
graf_demand_TRA_combustible('ProductionByTechnology','PJ',1,'BUS',Solver_selec)
graf_demand_TRA_combustible('ProductionByTechnology','PJ',1,'TCK',Solver_selec)
graf_demand_TRA_combustible('ProductionByTechnology','PJ',1,'STT',Solver_selec)
graf_demand_TRA_combustible('ProductionByTechnology','PJ',1,'MIC',Solver_selec)

#### TERCIARIO

In [ ]:
###### Definir colores de los combustibles

COLORES_GRUPOS = {
    'NGS': '#1f77b4',
    'JET': '#ff7f0e',
    'BGS': '#2ca02c',
    'BDL': '#d62728',
    'WAS': '#9467bd',
    'WOO': '#8c564b',
    'GSL': '#e377c2',
    'COA': '#7f7f7f',
    'ELC': '#bcbd22',
    'BAG': '#17becf',
    'DSL': '#aec7e8',
    'LPG': '#ffbb78',
    'PHEV': '#98df8a',
    'HEV': '#ff9896',
}

    # Agrupar por 'ELC'/ JET en la columna COLOR

def asignar_grupo(nombre):
    if 'ELC' in nombre:
        return 'ELC'
    elif 'JET' in nombre:
        return 'JET'
    elif 'LPG' in nombre:
        return 'LPG'
    elif 'BDL' in nombre:
        return 'BDL'
    elif 'WAS' in nombre:
        return 'WAS'
    elif 'BGS' in nombre:
        return 'BGS'
    elif 'GSL' in nombre:
        return 'GSL'
    elif 'DSL' in nombre:
        return 'DSL'
    elif 'NGS' in nombre:
        return 'NGS'
    elif 'BAG' in nombre:
        return 'BAG'
    elif 'COA' in nombre:
        return 'COA'
    elif 'FOL' in nombre:
        return 'FOL'
    elif 'APHEV' in nombre:
        return 'PHEV'
    elif 'AHEV' in nombre:
        return 'HEV' 
    else:
        return nombre



def graf_demand_TER_combustible_TOTAL(variable_name,UN,pasos,solver_selec):

### Definir dataframe a graficar según solver
    if solver_selec == "highs":
        df_var = sol_variable_to_df(
            sol,
            variable_name,
            ['REGION', 'TIMESLICE', 'TECHNOLOGY', 'FUEL', 'YEAR']
        ).drop(columns="DUMMY")
        df_var.head(50)
    elif solver_selec == "glpk":
        variable_mapping = {
            # 'RateOfProductionByTechnology': instance.RateOfProductionByTechnology,
            # 'RateOfUseByTechnology': instance.RateOfUseByTechnology,
            'ProductionByTechnology': instance.ProductionByTechnology,
            'UseByTechnology': instance.UseByTechnology
        }
        if variable_name not in variable_mapping:
            raise ValueError(f"Variable no válida: {variable_name}")
        variable = variable_mapping[variable_name]
        df_var = variable_to_dataframe(variable, index_names=['REGION', 'TIMESLICE', 'TECHNOLOGY', 'FUEL', 'YEAR']) 
        df_cap = df_var.copy()

    if df_cap.empty:
        print(f"No hay datos para la variable {variable_name}")
        return

    df_cap = df_cap[df_cap['TECHNOLOGY'].str.startswith('DEMTER')]


    if pasos ==1:
        df_cap['TIME'] = df_cap['YEAR'].astype(str)
    else:
        # Combining YEAR and TIMESLICE into TIME
        df_cap['TIME'] = df_cap['YEAR'].astype(str) + '_'+ df_cap['TIMESLICE']

    df_cap['COLOR'] = df_cap['TECHNOLOGY'] + '_' + df_cap['FUEL']

    df_cap['GROUP'] = df_cap['COLOR'].apply(asignar_grupo)

    df_final = df_cap.groupby(['GROUP', 'TIME'], as_index=False)['VALUE'].sum()
    df_final = df_final.rename(columns={'GROUP': 'COLOR'})

    df_final = df_final[df_final.groupby('COLOR')['VALUE'].transform('sum') > 1e-5]
    if UN=='TWh':
        df_final["VALUE"] /= 3.6  # Cambiar de PJ a TWh
    else:
        df_final["VALUE"] /= 1

    # Filtra solo los grupos presentes en df_final
    grupos_presentes = df_final["COLOR"].unique()

    # Construye la lista de colores en el orden correcto
    colores_final = [COLORES_GRUPOS.get(grupo, '#333333') for grupo in grupos_presentes]

    fig = px.bar(df_final, x="TIME", y="VALUE", color="COLOR",
                 title='SECTOR TERCIARIO - '+variable_name, labels={'VALUE': UN},
                 color_discrete_sequence=colores_final)
    # Ajustar la leyenda abajo
    # fig.update_layout(legend=dict(orientation="h", y=-0.4))
    fig.show()
    df_final.to_csv('resultados/'+'TERCIARIO_TOTAL_'+variable_name+'_'+UN+'.csv')

graf_demand_TER_combustible_TOTAL('ProductionByTechnology','PJ',1,Solver_selec)
graf_demand_TER_combustible_TOTAL('UseByTechnology','PJ',1,Solver_selec)

#### OTROS

In [ ]:
###### Definir colores de los combustibles

COLORES_GRUPOS = {
    'NGS': '#1f77b4',
    'JET': '#ff7f0e',
    'BGS': '#2ca02c',
    'BDL': '#d62728',
    'WAS': '#9467bd',
    'WOO': '#8c564b',
    'GSL': '#e377c2',
    'COA': '#7f7f7f',
    'ELC': '#bcbd22',
    'BAG': '#17becf',
    'DSL': '#aec7e8',
    'LPG': '#ffbb78',
    'PHEV': '#98df8a',
    'HEV': '#ff9896',
}

    # Agrupar por 'ELC'/ JET en la columna COLOR

def asignar_grupo(nombre):
    if 'ELC' in nombre:
        return 'ELC'
    elif 'JET' in nombre:
        return 'JET'
    elif 'LPG' in nombre:
        return 'LPG'
    elif 'BDL' in nombre:
        return 'BDL'
    elif 'WOO' in nombre:
        return 'WOO'
    elif 'BGS' in nombre:
        return 'BGS'
    elif 'GSL' in nombre:
        return 'GSL'
    elif 'DSL' in nombre:
        return 'DSL'
    elif 'NGS' in nombre:
        return 'NGS'
    elif 'BAG' in nombre:
        return 'BAG'
    elif 'COA' in nombre:
        return 'COA'
    elif 'FOL' in nombre:
        return 'FOL'
    elif 'APHEV' in nombre:
        return 'PHEV'
    elif 'AHEV' in nombre:
        return 'HEV' 
    else:
        return nombre



def graf_demand_OTROS_combustible_TOTAL(variable_name,UN,pasos,otro,solver_selec):
### Definir dataframe a graficar según solver
    if solver_selec == "highs":
        df_var = sol_variable_to_df(
            sol,
            variable_name,
            ['REGION', 'TIMESLICE', 'TECHNOLOGY', 'FUEL', 'YEAR']
        ).drop(columns="DUMMY")
        df_var.head(50)
    elif solver_selec == "glpk":
        variable_mapping = {
            # 'RateOfProductionByTechnology': instance.RateOfProductionByTechnology,
            # 'RateOfUseByTechnology': instance.RateOfUseByTechnology,
            'ProductionByTechnology': instance.ProductionByTechnology,
            'UseByTechnology': instance.UseByTechnology
        }
        if variable_name not in variable_mapping:
            raise ValueError(f"Variable no válida: {variable_name}")
        variable = variable_mapping[variable_name]
        df_var = variable_to_dataframe(variable, index_names=['REGION', 'TIMESLICE', 'TECHNOLOGY', 'FUEL', 'YEAR']) 
        df_cap = df_var.copy()

    if df_cap.empty:
        print(f"No hay datos para la variable {variable_name}")
        return

    df_cap = df_cap[df_cap['TECHNOLOGY'].str.startswith(otro)]


    if pasos ==1:
        df_cap['TIME'] = df_cap['YEAR'].astype(str)
    else:
        # Combining YEAR and TIMESLICE into TIME
        df_cap['TIME'] = df_cap['YEAR'].astype(str) + '_'+ df_cap['TIMESLICE']

    df_cap['COLOR'] = df_cap['TECHNOLOGY'] + '_' + df_cap['FUEL']

    df_cap['GROUP'] = df_cap['COLOR'].apply(asignar_grupo)

    df_final = df_cap.groupby(['GROUP', 'TIME'], as_index=False)['VALUE'].sum()
    df_final = df_final.rename(columns={'GROUP': 'COLOR'})

    df_final = df_final[df_final.groupby('COLOR')['VALUE'].transform('sum') > 1e-5]
    if UN=='TWh':
        df_final["VALUE"] /= 3.6  # Cambiar de PJ a TWh
    else:
        df_final["VALUE"] /= 1

    # Filtra solo los grupos presentes en df_final
    grupos_presentes = df_final["COLOR"].unique()

    # Construye la lista de colores en el orden correcto
    colores_final = [COLORES_GRUPOS.get(grupo, '#333333') for grupo in grupos_presentes]

    # Construir título adecuado
    if otro == 'DEMCOQ':
        titulo = 'Coquerias - Consumo por combustible - Energía final' 
    elif otro == 'DEMAGF':
        titulo = 'Agricultura y Pesca - Consumo por combustible - Energía final'


    fig = px.bar(df_final, x="TIME", y="VALUE", color="COLOR",
                 title=titulo, labels={'VALUE': UN},
                 color_discrete_sequence=colores_final)
    # Ajustar la leyenda abajo
    # fig.update_layout(legend=dict(orientation="h", y=-0.4))
    fig.show()
    df_final.to_csv('resultados/'+'OTROS_'+variable_name+'_'+UN+'.csv')


graf_demand_OTROS_combustible_TOTAL('UseByTechnology','PJ',1,'DEMCOQ',Solver_selec)  ##Energía final
graf_demand_OTROS_combustible_TOTAL('UseByTechnology','PJ',1,'DEMAGF',Solver_selec)  ##Energía final

#### EMISIONES

In [ ]:
##############################FUNCION PARA GRAFICAS VARIABLES AnnualTechnologyEmission
###############################  and AnnualTechnologyEmissionPenaltyByEmission  ###############################


###############################GRAFICA AnnualTechnologyEmission###############################
# Sum of the AnnualTechnologyEmissionByMode over the modes of operation.

###############################GRAFICA AnnualTechnologyEmissionPenaltyByEmission###############################
# Undiscounted annual cost of emission e by technology t. It is a function of the AnnualTechnologyEmission and the parameter EmissionPenalty.


def graf_emissions(variable_name,solver_selec):
### Definir dataframe a graficar según solver
    if solver_selec == "highs":
        df_var = sol_variable_to_df(
            sol,
            variable_name,
            ['REGION', 'TECHNOLOGY', 'EMISSION', 'YEAR']
        ).drop(columns="DUMMY")
        df_var.head(50)
    elif solver_selec == "glpk":
        variable_mapping = {
            'AnnualTechnologyEmission': instance.AnnualTechnologyEmission,
            'AnnualTechnologyEmissionPenaltyByEmission': instance.AnnualTechnologyEmissionPenaltyByEmission,
        }
        if variable_name not in variable_mapping:
            raise ValueError(f"Variable no válida: {variable_name}")
        variable = variable_mapping[variable_name]
        df_var = variable_to_dataframe(variable, index_names=['REGION', 'TECHNOLOGY', 'EMISSION', 'YEAR']) 
        df_cap = df_var.copy()

    # # Combining YEAR and TIMESLICE into TIME
    # df_cap['TIME'] = df_cap['YEAR'].astype(str) + '_'+ df_cap['TIMESLICE']

    # # Combining TECHNOLOGY and FUEL into COLOR
    # df_cap['COLOR'] = df_cap['TECHNOLOGY'].astype(str) + '_'+ df_cap['FUEL']

    # # Grouping data by COLOR and TIME
    # df_final = df_cap.groupby(['COLOR', 'TIME']).sum().reset_index()

    # Filtering out rows where VALUE sum is less than or equal to 1e-5
    df_final = df_cap[df_cap['VALUE'] > 1e-5]

    # Plotting with Plotly

    fig = px.bar(df_final, x="YEAR", y="VALUE", color="TECHNOLOGY", title=variable_name, labels={'VALUE':'Mt CO2e'})
    fig.show()
    df_final.to_csv('resultados/'+'EMISIONES_'+variable_name+'.csv')
###############################GRAFICA AnnualTechnologyEmission###############################
#Use of fuel f by technology t in time slice l

graf_emissions('AnnualTechnologyEmission',Solver_selec)

#### ALMACENAMIENTO

In [ ]:
if Correr=="Almacenamiento":

##############################FUNCION PARA GRAFICAS VARIABLES DE CAPACIDAD ALMACENAMIENTO###############################

    def graf_CAP_storage(variable_name,solver_selec):

        if solver_selec == "highs":
            df_var = sol_variable_to_df(
                sol,
                variable_name,
                ['REGION','STORAGE','YEAR']
            ).drop(columns="DUMMY")
            df_var.head(50)
        elif solver_selec == "glpk":
            variable_mapping = {
                'StorageUpperLimit': instance.StorageUpperLimit,
            }
            if variable_name not in variable_mapping:
                raise ValueError(f"Variable no válida: {variable_name}")
            variable = variable_mapping[variable_name]
            df_var = variable_to_dataframe(variable, index_names=['REGION','STORAGE','YEAR']) 
            df_cap = df_var.copy()

        # Grouping data by TECHNOLOGY and TIME
        df_final = df_cap.groupby(['STORAGE','YEAR']).sum().reset_index()

        # Filtering out rows where VALUE sum is less than or equal to 0
        df_final = df_final[df_final.groupby('STORAGE')['VALUE'].transform('sum') > 1e-5]

        # Plotting with Plotly
        fig = px.area(df_final, x="YEAR", y="VALUE", color="STORAGE", title=variable_name)
        fig.show()

        fig = px.bar(df_final, x="YEAR", y="VALUE", color="STORAGE", title=variable_name, labels={'VALUE':'PJ'})
        fig.show()


    graf_CAP_storage('StorageUpperLimit', Solver_selec)



    def graf_Op_Storage(variable_name,solver_selec):
        if solver_selec == "highs":
            df_var = sol_variable_to_df(
                sol,
                variable_name,
                ['REGION', 'STORAGE','SEASON','DAYTYPE', 'DAILYTIMEBRACKET','YEAR']
            ).drop(columns="DUMMY")
            df_var.head(50)
        elif solver_selec == "glpk":
            variable_mapping = {
                'NetChargeWithinYear': instance.NetChargeWithinYear,
                'NetChargeWithinDay': instance.NetChargeWithinDay,
                'RateOfStorageCharge': instance.RateOfStorageCharge,
                'RateOfStorageDischarge': instance.RateOfStorageDischarge,
            }
            if variable_name not in variable_mapping:
                raise ValueError(f"Variable no válida: {variable_name}")
            variable = variable_mapping[variable_name]
            df_var = variable_to_dataframe(variable, index_names=['REGION', 'STORAGE','SEASON','DAYTYPE', 'DAILYTIMEBRACKET','YEAR']) 
            df_cap = df_var.copy()

        # Creating DataFrame from the variable
        df_cap = pd.DataFrame([(r,s,ls,ld,lh,y, variable[(r,s,ls,ld,lh,y)].value) for (r,s,ls,ld,lh,y) in variable],
                            columns=['REGION', 'STORAGE','SEASON','DAYTYPE', 'DAILYTIMEBRACKET','YEAR', 'VALUE'])

        # Combining YEAR and TIMESLICE into TIME
        # df_cap['TIME'] = df_cap['YEAR'].astype(str) + '_'+ df_cap['SEASON'].astype(str)+ '_'+ df_cap['DAYTYPE'].astype(str)+ df_cap['DAILYTIMEBRACKET'].astype(str)
        # df_cap['TIME'] = df_cap['YEAR'].astype(str) + '_'+ df_cap['DAILYTIMEBRACKET'].astype(str)
        # Combining YEAR and TIMESLICE into TIME
        df_cap['TIME'] = df_cap['YEAR'].astype(str) + '_'+df_cap['DAILYTIMEBRACKET'].astype(str)

        df_cap = df_cap.sort_values(by='TIME', ascending=True)


        # Grouping data by COLOR and TIME
        df_final =df_cap

        # Plotting with Plotly
        fig = px.area(df_final, x="TIME", y="VALUE", color="STORAGE", title=variable_name)
        fig.show()

    graf_Op_Storage('NetChargeWithinYear', Solver_selec)
    graf_Op_Storage('NetChargeWithinDay', Solver_selec)
    graf_Op_Storage('RateOfStorageCharge', Solver_selec)
    graf_Op_Storage('RateOfStorageDischarge', Solver_selec)



    def graf_Op_Storage_SOC(variable_name,solver_selec):
        if solver_selec == "highs":
            df_var = sol_variable_to_df(
                sol,
                variable_name,
                ['REGION', 'STORAGE','YEAR']
            ).drop(columns="DUMMY")
            df_var.head(50)
        elif solver_selec == "glpk":
            variable_mapping = {
                'StorageLevelYearStart': instance.StorageLevelYearStart,
                'StorageLevelYearFinish': instance.StorageLevelYearFinish,
                'StorageLowerLimit': instance.StorageLowerLimit,
                'StorageUpperLimit': instance.StorageUpperLimit,
                'SalvageValueStorage': instance.SalvageValueStorage,
                'DiscountedSalvageValueStorage': instance.DiscountedSalvageValueStorage,      
                'TotalDiscountedStorageCost': instance.TotalDiscountedStorageCost,   
                'NewStorageCapacity': instance.NewStorageCapacity,      
                'CapitalInvestmentStorage': instance.CapitalInvestmentStorage,   
                'AccumulatedNewStorageCapacity': instance.AccumulatedNewStorageCapacity,   
            }
            if variable_name not in variable_mapping:
                raise ValueError(f"Variable no válida: {variable_name}")
            variable = variable_mapping[variable_name]
            df_var = variable_to_dataframe(variable, index_names=['REGION', 'STORAGE','YEAR']) 
            df_cap = df_var.copy()

        # Grouping data by COLOR and TIME
        df_final =df_cap

        # Plotting with Plotly
        fig = px.area(df_final, x="TIME", y="VALUE", color="STORAGE", title=variable_name)
        fig.show()

    graf_Op_Storage_SOC('StorageLevelYearStart', Solver_selec)
    graf_Op_Storage_SOC('StorageLevelYearFinish', Solver_selec)
    graf_Op_Storage_SOC('StorageLowerLimit', Solver_selec)
    graf_Op_Storage_SOC('StorageUpperLimit', Solver_selec)
    graf_Op_Storage_SOC('SalvageValueStorage', Solver_selec)
    graf_Op_Storage_SOC('DiscountedSalvageValueStorage', Solver_selec)
    graf_Op_Storage_SOC('TotalDiscountedStorageCost', Solver_selec)
    graf_Op_Storage_SOC('NewStorageCapacity', Solver_selec)
    graf_Op_Storage_SOC('CapitalInvestmentStorage', Solver_selec)
    graf_Op_Storage_SOC('AccumulatedNewStorageCapacity', Solver_selec)

## GRAPHS FROM EXTERNAL HIGHS SOLUTION FILE
Read an existing HiGHS solution file, parse its indices using the CSV sets, and generate all graphs — without needing to re-run the model or have a live Pyomo instance.

In [ ]:
# =============================================================================
# STEP 1 — Read HiGHS solution file and load input sets from CSV
# =============================================================================
# Uses:
#   sol_file_name  — defined in the Configuration cell
#   path_csv       — defined in the Configuration cell

import re
import os
import pandas as pd

# ── 1a. Parse the raw solution file ───────────────────────────────────────────
def _read_highs_solfile(filepath):
    """
    Parses a HiGHS human-readable solution file produced by:
        h.writeSolution("file.txt", 1)

    Each primal variable line has the format:
        VarName(index_str) numeric_value

    Returns a DataFrame with columns: VarName | IndexStr | Value.
    Reads only the 'Columns' block; stops at '# Rows' (duals not needed).
    """
    rows = []
    in_columns = False
    _re = re.compile(r'^(\w+)\((.+)\)\s+([\-\d\.eE\+]+)\s*$')

    with open(filepath, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.rstrip()
            if line.startswith("# Columns"):
                in_columns = True
                continue
            if in_columns and line.startswith("# Rows"):
                break
            if not in_columns or not line:
                continue
            m = _re.match(line)
            if m:
                rows.append((m.group(1), m.group(2), float(m.group(3))))

    return pd.DataFrame(rows, columns=["VarName", "IndexStr", "Value"])

print(f"Reading solution file: {sol_file_name}")
df_sol_raw = _read_highs_solfile(sol_file_name)
print(f"  -> {len(df_sol_raw):,} primal variable entries loaded")
print("  Variables found:", df_sol_raw["VarName"].value_counts().head(15).to_dict())

# ── 1b. Load model sets from CSV files ────────────────────────────────────────
def _load_set(folder, filename):
    fp = os.path.join(folder, filename)
    if os.path.exists(fp):
        return set(pd.read_csv(fp)["VALUE"].astype(str).str.strip())
    return set()

_tech_set     = _load_set(path_csv, "TECHNOLOGY.csv")
_fuel_set     = _load_set(path_csv, "FUEL.csv")
_region_set   = _load_set(path_csv, "REGION.csv")
_ts_set       = _load_set(path_csv, "TIMESLICE.csv")
_year_set     = _load_set(path_csv, "YEAR.csv")
_mode_set     = _load_set(path_csv, "MODE_OF_OPERATION.csv")
_emission_set = _load_set(path_csv, "EMISSION.csv")
_storage_set  = _load_set(path_csv, "STORAGE.csv")

print(f"\nSets loaded from: {path_csv}")
print(f"  Technologies      : {len(_tech_set)}")
print(f"  Fuels             : {len(_fuel_set)}")
print(f"  Regions           : {len(_region_set)}")
print(f"  Timeslices        : {len(_ts_set)}")
print(f"  Years             : {len(_year_set)}")
print(f"  Modes of operation: {len(_mode_set)}")
print(f"  Emissions         : {len(_emission_set)}")
print(f"  Storage           : {len(_storage_set)}")

# ── 1c. Auto-detect TIMESLICE values from RateOfActivity ──────────────────────
# Index structure: REGION _ TIMESLICE _ TECHNOLOGY _ MODE _ YEAR
_detected_ts = set()
_roa_rows = df_sol_raw[df_sol_raw["VarName"] == "RateOfActivity"]["IndexStr"]
for _idx_str in _roa_rows:
    _tokens = _idx_str.split("_")
    if len(_tokens) < 4:
        continue
    if not (re.match(r'^\d{4}$', _tokens[-1]) and re.match(r'^\d+$', _tokens[-2])):
        continue
    _core = _tokens[:-2]
    _region_found = False
    for _L in range(min(4, len(_core)), 0, -1):
        if "_".join(_core[:_L]) in _region_set:
            _core = _core[_L:]
            _region_found = True
            break
    if not _region_found:
        continue
    for _L in range(min(6, len(_core)), 0, -1):
        if "_".join(_core[-_L:]) in _tech_set:
            _ts_cand = "_".join(_core[:-_L])
            if _ts_cand:
                _detected_ts.add(_ts_cand)
            break

_new_ts = _detected_ts - _ts_set
if _new_ts:
    print(f"  WARNING: Timeslices in solution not in TIMESLICE.csv: {_new_ts}")
    print(f"    -> Adding them to the parser label set automatically.")
    _ts_set.update(_new_ts)
else:
    print(f"  OK: All timeslices match TIMESLICE.csv: {_ts_set}")

# All known labels — sorted longest-first so greedy match picks the longest name
_all_labels = sorted(
    _tech_set | _fuel_set | _region_set | _ts_set |
    _year_set | _mode_set | _emission_set | _storage_set,
    key=len, reverse=True)
_label_set = set(_all_labels)

# ── 1d. Parse index strings into tuples ────────────────────────────────────────
def _parse_index(index_str):
    """
    Greedily splits an underscore-joined index string into known-set tokens.
    Longest match wins at each position.
    e.g. 'RE1_TS_0_IN_DEMOTHDSL_1_2022' -> ('RE1', 'TS_0', 'IN_DEMOTHDSL', '1', '2022')
    """
    tokens = index_str.split("_")
    idx = []
    i = 0
    while i < len(tokens):
        matched = False
        for L in range(min(6, len(tokens) - i), 0, -1):
            candidate = "_".join(tokens[i : i + L])
            if candidate in _label_set:
                idx.append(candidate)
                i += L
                matched = True
                break
        if not matched:
            idx.append(tokens[i])
            i += 1
    return tuple(idx) if len(idx) > 1 else idx[0]

# ── 1e. Build sol_ext dict: sol_ext[VarName][index_tuple] = value ─────────────
sol_ext = {}
for vn, ixs, val in zip(df_sol_raw["VarName"],
                         df_sol_raw["IndexStr"],
                         df_sol_raw["Value"]):
    idx = _parse_index(ixs)
    if vn not in sol_ext:
        sol_ext[vn] = {}
    sol_ext[vn][idx] = val

print(f"\nsol_ext built with {len(sol_ext)} variables.")
for v in list(sol_ext.keys())[:6]:
    _first = next(iter(sol_ext[v]))
    print(f"  {v}  ->  sample key: {_first}  =  {sol_ext[v][_first]:.4g}")

In [ ]:
# =============================================================================
# STEP 2 — Build result DataFrames + compute derived variables
#
# The HiGHS solution stores only decision variables (RateOfActivity,
# NewCapacity, emission aggregates, etc.).  The following must be
# recomputed here from the CSV parameters:
#
#   ProductionByTechnology  = RateOfActivity x OutputActivityRatio x YearSplit
#   UseByTechnology         = RateOfActivity x InputActivityRatio  x YearSplit
#   AccumulatedNewCapacity  = sum NewCapacity(yy)  where 0 <= y-yy < OperLife
#   TotalCapacityAnnual     = AccumulatedNewCapacity + ResidualCapacity
# =============================================================================
import os
import pandas as pd

def sol_variable_to_df_ext(varname, dimnames, sol=None):
    """
    Converts sol_ext[varname] -> pd.DataFrame with columns (dimnames + VALUE).
    Uses sol_ext by default; pass sol= to override.
    """
    _sol = sol if sol is not None else sol_ext
    if varname not in _sol:
        print(f"  WARNING: '{varname}' not in solution — will be computed from CSV.")
        return pd.DataFrame(columns=dimnames + ["VALUE"])
    data = []
    for idx, val in _sol[varname].items():
        if not isinstance(idx, tuple):
            idx = (idx,)
        idx_ext = list(idx) + [None] * (len(dimnames) - len(idx))
        row = {dimnames[k]: idx_ext[k] for k in range(len(dimnames))}
        row["VALUE"] = float(val) if val is not None else 0.0
        data.append(row)
    df = pd.DataFrame(data)
    if "YEAR" in df.columns:
        df["YEAR"] = pd.to_numeric(df["YEAR"], errors="coerce").fillna(0).astype(int)
    return df

# ── A. Direct decision variables ───────────────────────────────────────────────
print("Building DataFrames from solution ...\n")

df_ext_NewCapacity = sol_variable_to_df_ext(
    "NewCapacity", ["REGION", "TECHNOLOGY", "YEAR"])

df_ext_Emissions = sol_variable_to_df_ext(
    "AnnualTechnologyEmission", ["REGION", "TECHNOLOGY", "EMISSION", "YEAR"])

df_ext_StorageUpperLimit = sol_variable_to_df_ext(
    "StorageUpperLimit", ["REGION", "STORAGE", "YEAR"])

# ── B. Load CSV parameters needed for derived computations ─────────────────────
print("Loading CSV parameters ...")
df_oar = pd.read_csv(os.path.join(path_csv, "OutputActivityRatio.csv"))
df_iar = pd.read_csv(os.path.join(path_csv, "InputActivityRatio.csv"))
df_ys  = pd.read_csv(os.path.join(path_csv, "YearSplit.csv"))
df_rc  = pd.read_csv(os.path.join(path_csv, "ResidualCapacity.csv"))
df_ol  = pd.read_csv(os.path.join(path_csv, "OperationalLife.csv"))

# Normalise dtypes
for _df in (df_oar, df_iar):
    for c in ["REGION", "TECHNOLOGY", "MODE_OF_OPERATION", "FUEL"]:
        if c in _df.columns:
            _df[c] = _df[c].astype(str).str.strip()
    _df["YEAR"] = _df["YEAR"].astype(int)
    _df.rename(columns={"VALUE": "RATIO"}, inplace=True)

df_ys["TIMESLICE"] = df_ys["TIMESLICE"].astype(str).str.strip()
df_ys["YEAR"]      = df_ys["YEAR"].astype(int)
df_rc["REGION"]     = df_rc["REGION"].astype(str).str.strip()
df_rc["TECHNOLOGY"] = df_rc["TECHNOLOGY"].astype(str).str.strip()
df_rc["YEAR"]       = df_rc["YEAR"].astype(int)
df_rc.rename(columns={"VALUE": "RC"}, inplace=True)

# ── C. ProductionByTechnology and UseByTechnology ──────────────────────────────
print("Building RateOfActivity DataFrame ...")
df_roa = sol_variable_to_df_ext(
    "RateOfActivity",
    ["REGION", "TIMESLICE", "TECHNOLOGY", "MODE_OF_OPERATION", "YEAR"])
df_roa = df_roa[df_roa["VALUE"].abs() > 1e-9]
print(f"  RateOfActivity non-zero rows: {len(df_roa):,}")

for c in ["REGION", "TIMESLICE", "TECHNOLOGY", "MODE_OF_OPERATION"]:
    df_roa[c] = df_roa[c].astype(str).str.strip()
df_roa["YEAR"] = df_roa["YEAR"].astype(int)

# Attach YearSplit
df_roa = df_roa.merge(
    df_ys.rename(columns={"VALUE": "YS"}), on=["TIMESLICE", "YEAR"], how="left")
df_roa["YS"]      = df_roa["YS"].fillna(1.0)
df_roa["RATE_YS"] = df_roa["VALUE"] * df_roa["YS"]

print("\nComputing ProductionByTechnology ...")
df_prod = df_roa.merge(
    df_oar[["REGION", "TECHNOLOGY", "FUEL", "MODE_OF_OPERATION", "YEAR", "RATIO"]],
    on=["REGION", "TECHNOLOGY", "MODE_OF_OPERATION", "YEAR"], how="inner")
df_prod["VALUE"] = df_prod["RATE_YS"] * df_prod["RATIO"]
df_ext_Production = (
    df_prod
    .groupby(["REGION", "TIMESLICE", "TECHNOLOGY", "FUEL", "YEAR"], as_index=False)["VALUE"]
    .sum())
df_ext_Production = df_ext_Production[df_ext_Production["VALUE"].abs() > 1e-9]
print(f"  ProductionByTechnology rows: {len(df_ext_Production):,}")

print("Computing UseByTechnology ...")
df_use = df_roa.merge(
    df_iar[["REGION", "TECHNOLOGY", "FUEL", "MODE_OF_OPERATION", "YEAR", "RATIO"]],
    on=["REGION", "TECHNOLOGY", "MODE_OF_OPERATION", "YEAR"], how="inner")
df_use["VALUE"] = df_use["RATE_YS"] * df_use["RATIO"]
df_ext_Use = (
    df_use
    .groupby(["REGION", "TIMESLICE", "TECHNOLOGY", "FUEL", "YEAR"], as_index=False)["VALUE"]
    .sum())
df_ext_Use = df_ext_Use[df_ext_Use["VALUE"].abs() > 1e-9]
print(f"  UseByTechnology rows: {len(df_ext_Use):,}")

# ── D. AccumulatedNewCapacity and TotalCapacityAnnual ──────────────────────────
print("\nComputing AccumulatedNewCapacity and TotalCapacityAnnual ...")
df_nc = df_ext_NewCapacity[df_ext_NewCapacity["VALUE"].abs() > 1e-9].copy()
df_nc["YEAR"]       = df_nc["YEAR"].astype(int)
df_nc["REGION"]     = df_nc["REGION"].astype(str).str.strip()
df_nc["TECHNOLOGY"] = df_nc["TECHNOLOGY"].astype(str).str.strip()

_ol_dict = {(str(r).strip(), str(t).strip()): int(v)
            for r, t, v in zip(df_ol["REGION"], df_ol["TECHNOLOGY"], df_ol["VALUE"])}

_all_years     = sorted(df_nc["YEAR"].unique())
_accum_records = []
for (region, tech), grp in df_nc.groupby(["REGION", "TECHNOLOGY"]):
    ol = _ol_dict.get((region, tech), 0)
    if ol == 0:
        continue
    for row in grp.itertuples(index=False):
        yy  = int(row.YEAR)
        val = float(row.VALUE)
        for y in _all_years:
            if 0 <= y - yy < ol:
                _accum_records.append((region, tech, y, val))

df_ext_AccumCapacity = (
    pd.DataFrame(_accum_records, columns=["REGION", "TECHNOLOGY", "YEAR", "VALUE"])
    .groupby(["REGION", "TECHNOLOGY", "YEAR"], as_index=False)["VALUE"]
    .sum())
print(f"  AccumulatedNewCapacity rows: {len(df_ext_AccumCapacity):,}")

df_ext_TotalCapacity = (
    df_ext_AccumCapacity
    .merge(df_rc, on=["REGION", "TECHNOLOGY", "YEAR"], how="outer")
    .fillna(0)
    .assign(VALUE=lambda d: d["VALUE"] + d["RC"])
    [["REGION", "TECHNOLOGY", "YEAR", "VALUE"]])
df_ext_TotalCapacity = df_ext_TotalCapacity[df_ext_TotalCapacity["VALUE"].abs() > 1e-9]
print(f"  TotalCapacityAnnual rows: {len(df_ext_TotalCapacity):,}")

# ── E. Summary ─────────────────────────────────────────────────────────────────
print("\n── Summary ──────────────────────────────────────────────")
for name, df in [
    ("TotalCapacityAnnual",      df_ext_TotalCapacity),
    ("NewCapacity",              df_ext_NewCapacity),
    ("AccumulatedNewCapacity",   df_ext_AccumCapacity),
    ("ProductionByTechnology",   df_ext_Production),
    ("UseByTechnology",          df_ext_Use),
    ("AnnualTechnologyEmission", df_ext_Emissions),
]:
    nz = df[df["VALUE"].abs() > 1e-9] if not df.empty else df
    print(f"  {name:35s}: {len(df):>8,} rows  ({len(nz):,} non-zero)")

# ── F. Save all result tables to tables_dir ────────────────────────────────────
print(f"\nSaving tables to: {tables_dir}")
_tables = {
    "TotalCapacityAnnual.csv"     : df_ext_TotalCapacity,
    "NewCapacity.csv"             : df_ext_NewCapacity,
    "AccumulatedNewCapacity.csv"  : df_ext_AccumCapacity,
    "ProductionByTechnology.csv"  : df_ext_Production,
    "UseByTechnology.csv"         : df_ext_Use,
    "AnnualTechnologyEmission.csv": df_ext_Emissions,
}
for fname, df in _tables.items():
    df.to_csv(os.path.join(str(tables_dir), fname), index=False)
    print(f"  Saved: {fname}")
print("\nAll DataFrames ready for graphing.")

In [ ]:
# =============================================================================
# STEP 3 — Power sector graphs (Capacity + Production)
# =============================================================================
import colorsys
import os
import plotly.express as px

# ── Technology colour map ──────────────────────────────────────────────────────
_FAMILIAS_TEC = {
    "SOLAR"          : ["PWRSOLRTP", "PWRSOLRTP_ZNI", "PWRSOLUGE",
                         "PWRSOLUGE_BAT", "PWRSOLUPE"],
    "HYDRO"          : ["PWRHYDDAM", "PWRHYDROR", "PWRHYDROR_NDC"],
    "WIND"           : ["PWRWNDONS", "PWRWNDOFS_FIX", "PWRWNDOFS_FLO"],
    "THERMAL_FOSSIL" : ["PWRCOA", "PWRCOACCS", "PWRNGS_CC", "PWRNGS_CS",
                         "PWRNGSCCS", "PWRDSL", "PWRFOIL", "PWRJET", "PWRLPG"],
    "NUCLEAR"        : ["PWRNUC"],
    "BIOMASS_WASTE"  : ["PWRAFR", "PWRBGS", "PWRWAS"],
    "OTHER"          : ["PWRCSP", "PWRGEO", "PWRSTD"],
}
_COLOR_BASE = {
    "SOLAR"         : "#FDB813",
    "HYDRO"         : "#1F77B4",
    "WIND"          : "#2CA02C",
    "THERMAL_FOSSIL": "#2B2B2B",
    "NUCLEAR"       : "#7B3F98",
    "BIOMASS_WASTE" : "#8C6D31",
    "OTHER"         : "#17BECF",
}

def _gen_shades(color_hex, n):
    """Generate n shades from a base hex colour.\"""
    r, g, b = tuple(int(color_hex.lstrip("#")[i:i+2], 16) / 255 for i in (0, 2, 4))
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    shades = []
    for i in range(n):
        li = 0.35 + 0.35 * i / max(1, n - 1)
        si = s if s < 0.2 else min(1.0, s * 1.05)
        ri, gi, bi = colorsys.hls_to_rgb(h, li, si)
        shades.append(f"#{int(ri*255):02x}{int(gi*255):02x}{int(bi*255):02x}")
    return shades

_COLOR_MAP_PWR = {}
for _fam, _techs in _FAMILIAS_TEC.items():
    for _tech, _color in zip(_techs, _gen_shades(_COLOR_BASE[_fam], len(_techs))):
        _COLOR_MAP_PWR[_tech] = _color

# ── Helper — capacity bar chart ────────────────────────────────────────────────
def _graf_cap_ext(df_in, variable_label, tech_prefix="PWR", unit="GW",
                  color_map=None):
    df = df_in[df_in["VALUE"].abs() > 1e-5].copy()
    if tech_prefix:
        df = df[df["TECHNOLOGY"].str.startswith(tech_prefix)]
    df = df.groupby(["TECHNOLOGY", "YEAR"], as_index=False)["VALUE"].sum()
    if unit == "GW":
        df["VALUE"] /= 31.536

    fig = px.bar(df, x="YEAR", y="VALUE", color="TECHNOLOGY",
                 title=f"Power Sector – {variable_label}",
                 labels={"VALUE": unit},
                 color_discrete_map=color_map or {})
    fig.update_layout(legend=dict(traceorder="normal"))
    fig.show()

    csv_path  = os.path.join(str(tables_dir), f"Cap_Elec_{variable_label}.csv")
    html_path = os.path.join(str(graphs_dir), f"Cap_Elec_{variable_label}.html")
    df.to_csv(csv_path, index=False)
    fig.write_html(html_path)
    print(f"  Table: {csv_path}")
    print(f"  Graph: {html_path}")

# ── Helper — production bar chart ─────────────────────────────────────────────
def _graf_prod_ext(df_in, variable_label, tech_prefix="PWR", unit="TWh",
                   color_map=None, div=1):
    df = df_in[df_in["TECHNOLOGY"].str.startswith(tech_prefix)].copy()
    df = df[df["VALUE"].abs() > 1e-5]
    df["YEAR"] = df["YEAR"].astype(int)

    if div == 1:
        df["TIME"]     = df["YEAR"].astype(str)
        df["TIME_NUM"] = df["YEAR"]
    else:
        df["BLOCK"]    = df["TIMESLICE"].str.extract(r'(\d+)$').astype(int)
        df["TIME"]     = df["YEAR"].astype(str) + "_" + df["TIMESLICE"]
        df["TIME_NUM"] = df["YEAR"] + (df["BLOCK"] - 1) / df["BLOCK"].max()

    df_final = df.groupby(["TECHNOLOGY", "TIME", "TIME_NUM"], as_index=False)["VALUE"].sum()
    df_final = df_final[df_final["VALUE"].abs() > 1e-5]
    if unit == "TWh":
        df_final["VALUE"] /= 3.6
    df_final = df_final.sort_values("TIME_NUM")

    fig = px.bar(df_final, x="TIME", y="VALUE", color="TECHNOLOGY",
                 title=f"Electricity Generation – {variable_label}",
                 labels={"VALUE": unit},
                 color_discrete_map=color_map or {})
    fig.update_xaxes(categoryorder="array", categoryarray=df_final["TIME"].tolist())
    fig.update_layout(legend=dict(traceorder="normal"))
    fig.show()

    csv_path  = os.path.join(str(tables_dir), f"Prod_Elec_{variable_label}_{div}TS.csv")
    html_path = os.path.join(str(graphs_dir), f"Prod_Elec_{variable_label}_{div}TS.html")
    df_final.to_csv(csv_path, index=False)
    fig.write_html(html_path)
    print(f"  Table: {csv_path}")
    print(f"  Graph: {html_path}")

# ── Run power sector graphs ────────────────────────────────────────────────────
print("=== Power Sector — Capacity ===")
_graf_cap_ext(df_ext_TotalCapacity,  "TotalCapacityAnnual",    color_map=_COLOR_MAP_PWR)
_graf_cap_ext(df_ext_NewCapacity,    "NewCapacity",            color_map=_COLOR_MAP_PWR)
_graf_cap_ext(df_ext_AccumCapacity,  "AccumulatedNewCapacity", color_map=_COLOR_MAP_PWR)

print("\n=== Power Sector — Production ===")
_graf_prod_ext(df_ext_Production, "ProductionByTechnology", div=div, color_map=_COLOR_MAP_PWR)

In [ ]:
# =============================================================================
# STEP 4 — Demand sector graphs + Emissions
# =============================================================================
import os
import plotly.express as px

_FUEL_COLORS = {
    "NGS": "#1f77b4", "JET": "#ff7f0e", "BGS": "#2ca02c", "BDL": "#d62728",
    "WAS": "#9467bd", "WOO": "#8c564b", "GSL": "#e377c2", "COA": "#7f7f7f",
    "ELC": "#bcbd22", "BAG": "#17becf", "DSL": "#aec7e8", "LPG": "#ffbb78",
    "FOL": "#98df8a", "PHEV": "#ff9896", "HEV": "#ccaabb", "AUT": "#aabbcc",
}
_FUEL_KEY_ORDER = ["ELC", "NGS", "DSL", "GSL", "LPG", "JET", "BDL", "BGS",
                   "WAS", "WOO", "BAG", "COA", "FOL", "PHEV", "HEV", "AUT"]

def _assign_fuel_group(name):
    for g in _FUEL_KEY_ORDER:
        if g in name:
            return g
    return name

def _graf_sector_ext(df_activity, variable_label, tech_startswith,
                     tech_contains=None, aggregate_by_fuel=True,
                     unit="PJ", div=1, title_prefix=""):
    """
    Generic demand-sector bar chart for ProductionByTechnology or UseByTechnology.

    Parameters
    ----------
    df_activity     : df_ext_Production or df_ext_Use
    variable_label  : string used in title and output filename
    tech_startswith : technology prefix filter (e.g. 'DEMRES', 'DEMIND')
    tech_contains   : optional secondary filter string
    aggregate_by_fuel : True -> group by fuel key; False -> keep technology name
    unit            : 'PJ', 'TWh', or 'Gpc'
    div             : number of time slices (1 = annual)
    title_prefix    : prefix for chart title
    """
    df = df_activity.copy()
    df = df[df["TECHNOLOGY"].str.startswith(tech_startswith)]
    if tech_contains:
        df = df[df["TECHNOLOGY"].str.contains(tech_contains)]
    if df.empty:
        print(f"  WARNING: No data for {tech_startswith}"
              f"{'/' + tech_contains if tech_contains else ''}")
        return

    df["TIME"] = (df["YEAR"].astype(str) if div == 1
                  else df["YEAR"].astype(str) + "_" + df["TIMESLICE"])

    df["COLOR"] = ((df["TECHNOLOGY"] + "_" + df["FUEL"]).apply(_assign_fuel_group)
                   if aggregate_by_fuel else df["TECHNOLOGY"])

    df_final = df.groupby(["COLOR", "TIME"], as_index=False)["VALUE"].sum()
    df_final = df_final[
        df_final.groupby("COLOR")["VALUE"].transform("sum").abs() > 1e-5]

    if unit == "TWh":
        df_final["VALUE"] /= 3.6
    elif unit == "Gpc":
        df_final["VALUE"] /= 1.0095581216

    grupos  = df_final["COLOR"].unique()
    colores = [_FUEL_COLORS.get(g, "#555555") for g in grupos]
    title   = f"{title_prefix} - {variable_label}" if title_prefix else variable_label

    fig = px.bar(df_final, x="TIME", y="VALUE", color="COLOR",
                 title=title, labels={"VALUE": unit},
                 color_discrete_sequence=colores)
    fig.show()

    _safe     = (tech_startswith + ("_" + tech_contains if tech_contains else "")).replace("/", "_")
    csv_path  = os.path.join(str(tables_dir), f"EXT_{_safe}_{variable_label}_{unit}.csv")
    html_path = os.path.join(str(graphs_dir), f"EXT_{_safe}_{variable_label}_{unit}.html")
    df_final.to_csv(csv_path, index=False)
    fig.write_html(html_path)
    print(f"  Table: {csv_path}")
    print(f"  Graph: {html_path}")

# ── Natural Gas ────────────────────────────────────────────────────────────────
print("=== Natural Gas (supply) ===")
_graf_sector_ext(df_ext_Production, "ProductionByTechnology",
                 "UPSREG", aggregate_by_fuel=False, unit="PJ",
                 title_prefix="Natural Gas Supply")
_graf_sector_ext(df_ext_Production, "ProductionByTechnology",
                 "MINNGS", aggregate_by_fuel=False, unit="PJ",
                 title_prefix="Natural Gas Mining")

print("\n=== Natural Gas (consumption) ===")
_df_gn = df_ext_Use[df_ext_Use["TECHNOLOGY"].str.contains("NGS")].copy()
_df_gn["TIME"]  = _df_gn["YEAR"].astype(str)
_df_gn["COLOR"] = _df_gn["TECHNOLOGY"]
_df_gn_f = _df_gn.groupby(["COLOR", "TIME"], as_index=False)["VALUE"].sum()
_df_gn_f = _df_gn_f[_df_gn_f["VALUE"].abs() > 1e-5]
fig_gn = px.bar(_df_gn_f, x="TIME", y="VALUE", color="COLOR",
                title="Natural Gas Consumption - UseByTechnology",
                labels={"VALUE": "PJ"})
fig_gn.show()
fig_gn.write_html(os.path.join(str(graphs_dir), "EXT_NGS_UseByTechnology_PJ.html"))
_df_gn_f.to_csv(os.path.join(str(tables_dir), "EXT_NGS_UseByTechnology_PJ.csv"), index=False)

# ── Refineries ─────────────────────────────────────────────────────────────────
print("\n=== Refineries ===")
_graf_sector_ext(df_ext_Production, "ProductionByTechnology",
                 "UPSREF", unit="PJ", title_prefix="Refineries - Production")
_graf_sector_ext(df_ext_Use, "UseByTechnology",
                 "UPSREF", unit="PJ", title_prefix="Refineries - Use")

# ── Residential sector ─────────────────────────────────────────────────────────
print("\n=== Residential (total) ===")
_graf_sector_ext(df_ext_Use, "UseByTechnology",
                 "DEMRES", unit="PJ", title_prefix="Residential - Total")
for _sub in ["CKN", "WHT", "AIR", "FAN", "REF", "TV", "WSH", "OTH"]:
    _graf_sector_ext(df_ext_Use, "UseByTechnology", "DEMRES",
                     tech_contains=_sub, unit="PJ",
                     title_prefix=f"Residential - {_sub}")

# ── Industrial sector ──────────────────────────────────────────────────────────
print("\n=== Industrial (total) ===")
_graf_sector_ext(df_ext_Use, "UseByTechnology",
                 "DEMIND", unit="PJ", title_prefix="Industrial - Total")
for _sub in ["BOI", "FUR", "MPW", "AIR", "REF", "ILU", "OTH"]:
    _graf_sector_ext(df_ext_Production, "ProductionByTechnology", "DEMIND",
                     tech_contains=_sub, unit="PJ",
                     title_prefix=f"Industrial - {_sub}")

# ── Transport sector ───────────────────────────────────────────────────────────
print("\n=== Transport (total) ===")
_graf_sector_ext(df_ext_Use,        "UseByTechnology",
                 "DEMTRA", unit="PJ", title_prefix="Transport - Total (use)")
_graf_sector_ext(df_ext_Production, "ProductionByTechnology",
                 "DEMTRA", unit="PJ", title_prefix="Transport - Total (production)")
for _sub in ["MOT", "LDV", "FWD", "BUS", "TCK", "STT", "MIC"]:
    _graf_sector_ext(df_ext_Production, "ProductionByTechnology", "DEMTRA",
                     tech_contains=_sub, unit="PJ",
                     title_prefix=f"Transport - {_sub}")

# ── Tertiary sector ────────────────────────────────────────────────────────────
print("\n=== Tertiary sector ===")
_graf_sector_ext(df_ext_Production, "ProductionByTechnology",
                 "DEMTER", unit="PJ", title_prefix="Tertiary - Production")
_graf_sector_ext(df_ext_Use,        "UseByTechnology",
                 "DEMTER", unit="PJ", title_prefix="Tertiary - Use")

# ── Other sectors ──────────────────────────────────────────────────────────────
print("\n=== Other sectors ===")
_graf_sector_ext(df_ext_Use, "UseByTechnology",
                 "DEMCOQ", unit="PJ", title_prefix="Coke plants")
_graf_sector_ext(df_ext_Use, "UseByTechnology",
                 "DEMAGF", unit="PJ", title_prefix="Agriculture and Fisheries")

# ── Emissions ──────────────────────────────────────────────────────────────────
print("\n=== Emissions ===")
_df_em = df_ext_Emissions[df_ext_Emissions["VALUE"].abs() > 1e-5].copy()
if not _df_em.empty:
    fig_em = px.bar(_df_em, x="YEAR", y="VALUE", color="TECHNOLOGY",
                    title="Annual Technology Emissions",
                    labels={"VALUE": "Mt CO2e"})
    fig_em.show()
    html_path_em = os.path.join(str(graphs_dir), "EXT_EMISIONES_AnnualTechnologyEmission.html")
    csv_path_em  = os.path.join(str(tables_dir), "EXT_EMISIONES_AnnualTechnologyEmission.csv")
    fig_em.write_html(html_path_em)
    _df_em.to_csv(csv_path_em, index=False)
    print(f"  Graph: {html_path_em}")
    print(f"  Table: {csv_path_em}")
else:
    print("  WARNING: No emission data found in solution.")